# 📚 Notebook 04 — Advanced Retrieval

> **Series**: Career AI Agent · LangChain Engineering Track  
> **Prerequisites**: Notebooks 01–03  
> **Scope**: Part 1 — Why Basic RAG Fails · Part 2 — MultiQueryRetriever

---

> **Session Note**  
> This notebook reconnects to the persisted Vector Store and LLM from Notebook 02.  
> It runs standalone in a fresh kernel — `llm`, `embedding_model`, `vector_db`, and `base_retriever` are rebuilt here from on-disk resources.


In [2]:
import os
import sys
from pathlib import Path

# Disable LangSmith tracing — it will be covered in a dedicated notebook
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_API_KEY"] = ""

# ── Project root resolution (robust: walks up looking for .git or src/) ───────
def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in [".git", "pyproject.toml", "setup.py", "src"]):
            return parent
    return start  # fallback

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model=settings.MODEL_NAME,
    api_key=settings.OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

# ── Embedding model (load from local cache — no download) ────────────────────
embedding_model = HuggingFaceEmbeddings(
    model_name=settings.EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# ── Reconnect to the EXISTING persisted Chroma vector store ──────────────────
VECTOR_DB_PATH = PROJECT_ROOT / settings.VECTOR_DB_PATH
vector_db = Chroma(
    persist_directory=str(VECTOR_DB_PATH),
    embedding_function=embedding_model,
)

# ── Base retriever (top-3 chunks by cosine similarity) ───────────────────────
base_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

print(f"✅ Connected to Vector DB at: {VECTOR_DB_PATH}")
print(f"   Documents in store: {vector_db._collection.count()}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2399.09it/s]


✅ Connected to Vector DB at: d:\career-ai-agent\storage\vector_db
   Documents in store: 285


---

# Part 1 — Why Basic RAG Fails

The naive RAG pipeline:

```
Query → Embed → Cosine Similarity → Top-k Chunks → LLM → Answer
```

This breaks reliably in production. Understanding exactly *where* is prerequisite knowledge before learning fixes.


## How Naive Similarity Search Works

```
User Query
    │ embed
    ▼
Query Vector ──► cosine similarity against every stored chunk vector
                         │
                    Rank by score
                         │
                    Top-k chunks returned
```

**Core assumption:** geometric closeness in embedding space = semantic relevance.  
This assumption fails more often than expected.


## Failure Modes

| # | Failure | Root Cause | Example |
|---|---|---|---|
| 1 | **Vocabulary Mismatch** | Same meaning, different words | "heart attack" vs "myocardial infarction" |
| 2 | **Synonym Blindness** | Embeddings don't perfectly cluster synonyms | "ML Engineer" vs "Machine Learning Specialist" |
| 3 | **Multi-hop Reasoning** | Answer requires bridging two separate chunks | "Mentor of the DeepMind founder?" |
| 4 | **Long Document Noise** | Relevant sentence buried in a large chunk | 1 key sentence + 490 noise words → low score |
| 5 | **Ambiguous Queries** | One string, multiple intents | "Python" → language? snake? Monty Python? |
| 6 | **Negation Blindness** | Embeddings don't model negation | "Jobs NOT requiring a degree" retrieves degree-requirement docs |
| 7 | **Top-k Rigidity** | Fixed k is never the right k | k=3 misses chunk 4; k=10 floods the context window |
| 8 | **Paraphrase Gap** | Query phrased differently from the document | "Get promoted?" vs "Career advancement strategies" |

---

**Vocabulary Mismatch — trace**
```
Query:  "What is the best way to find a job?"
Doc A:  "Effective strategies for career placement and employment acquisition."
        → Cosine sim: 0.71  ← LOW (same meaning, different words)
Doc B:  "Job search tips for beginners."
        → Cosine sim: 0.94  ← HIGH (surface word overlap wins)
```
Doc B wins on surface words. Semantically richer Doc A loses.

**Multi-hop — trace**
```
Question: "What Python libraries does the Senior ML Engineer role require?"
Chunk A (retrieved):     "The Senior ML Engineer role requires 3+ years of experience."
Chunk B (NOT retrieved): "Candidates must be proficient in TensorFlow, PyTorch, scikit-learn."
```
Chunk B never mentions "Senior ML Engineer" — the query vector never reaches it.

**Top-k Rigidity**
```
k=3  → Chunks 1,2,3. Chunk 4 has the answer. Missed.
k=10 → Chunks 1–10. Context window flooded. LLM "Lost in the Middle" kicks in.
```


## Live Demo — Observing Retrieval Failure

Three deliberately hard queries run through `base_retriever`. We inspect what gets retrieved and why it falls short.


In [3]:
# ── Helper: display retrieved documents clearly ──────────────────────────────
def display_retrieval_result(query: str, docs: list, label: str = "base_retriever") -> None:
    """Pretty-print retrieval results for inspection and debugging."""
    print(f"{'═'*65}")
    print(f"  Retriever : {label}")
    print(f"  Query     : {query}")
    print(f"  Retrieved : {len(docs)} chunk(s)")
    print(f"{'─'*65}")
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        page   = doc.metadata.get("page", "N/A")
        # Show first 200 chars of content — enough to judge relevance
        snippet = doc.page_content.replace("\n", " ").strip()[:200]
        print(f"  [{i}] Source : {source}  (page {page})")
        print(f"       Snippet: {snippet}...")
        print()
    print(f"{'═'*65}\n")


### Query A — Paraphrase Gap (vocabulary mismatch)

In [4]:
query_a = "What competencies are essential for machine learning practitioners?"

docs_a = base_retriever.invoke(query_a)
display_retrieval_result(query_a, docs_a)

# ── Analysis ─────────────────────────────────────────────────────────────────
print("🔍 Analysis:")
print("   If the dataset uses 'skills' or 'requirements' instead of 'competencies',")
print("   the retriever may miss the most relevant chunks.")
print("   This is Vocabulary Mismatch / Paraphrase Gap in action.")


═════════════════════════════════════════════════════════════════
  Retriever : base_retriever
  Query     : What competencies are essential for machine learning practitioners?
  Retrieved : 3 chunk(s)
─────────────────────────────────────────────────────────────────
  [1] Source : d:\career-ai-agent\data\resumes\ResumeGuideforStudentsGeneral.pdf  (page 12)
       Snippet: The UCC offers several options for getting help in creating and perfecting your resume.  Take advantage of  one or more of the following services available to students and alumni:      Career Coaching...

  [2] Source : d:\career-ai-agent\data\interview\10-41-Essential-Machine-Learning-Interview-Questions.pdf  (page 19)
       Snippet: Related to the last point, most organizations hiring for machine learn- ing positions will look for your formal experience in the field.  Research papers, co-authored or supervised by leaders in the f...

  [3] Source : d:\career-ai-agent\data\career\UMich_Alumni_Networking_Guide.pdf  

### Query B — Multi-hop Question

In [5]:
query_b = "What is the salary range for a role that requires Python and machine learning skills?"

docs_b = base_retriever.invoke(query_b)
display_retrieval_result(query_b, docs_b)

# ── Analysis ─────────────────────────────────────────────────────────────────
print("🔍 Analysis:")
print("   The answer requires finding a chunk about 'Python + ML skills'")
print("   AND a separate chunk about 'salary'. Top-3 similarity search")
print("   is unlikely to retrieve both simultaneously.")


═════════════════════════════════════════════════════════════════
  Retriever : base_retriever
  Query     : What is the salary range for a role that requires Python and machine learning skills?
  Retrieved : 3 chunk(s)
─────────────────────────────────────────────────────────────────
  [1] Source : d:\career-ai-agent\data\resumes\ResumeGuideforStudentsGeneral.pdf  (page 12)
       Snippet: development, including application documents, job search strategies, interviewing, and salary negotiation.    • Hire Red Raiders Resource Library: Within Hire Red Raiders, there is a library of resume...

  [2] Source : d:\career-ai-agent\data\resumes\ResumeGuideforStudentsGeneral.pdf  (page 12)
       Snippet: The UCC offers several options for getting help in creating and perfecting your resume.  Take advantage of  one or more of the following services available to students and alumni:      Career Coaching...

  [3] Source : d:\career-ai-agent\data\resumes\MASTERS-RESUME-GUIDE.pdf  (page 6)
      

### Query C — Ambiguous Query

In [6]:
query_c = "How do I advance?"

docs_c = base_retriever.invoke(query_c)
display_retrieval_result(query_c, docs_c)

# ── Analysis ─────────────────────────────────────────────────────────────────
print("🔍 Analysis:")
print("   'How do I advance?' could mean:")
print("   → Career advancement / promotion strategies")
print("   → Advanced technical skills to learn")
print("   → Advancing in a job application process")
print()
print("   A single embedding vector cannot capture all three intents at once.")
print("   The retriever picks one interpretation — possibly the wrong one.")


═════════════════════════════════════════════════════════════════
  Retriever : base_retriever
  Query     : How do I advance?
  Retrieved : 3 chunk(s)
─────────────────────────────────────────────────────────────────
  [1] Source : d:\career-ai-agent\data\interview\CMU_Behavioral_Interview_Guide.pdf  (page 0)
       Snippet: Before the Interview:   • Spend time thoroughly researching the company. You will likely be asked why you are  interested in the company, and you want to be able to connect your interests back  to the...

  [2] Source : d:\career-ai-agent\data\career\UMich_Alumni_Networking_Guide.pdf  (page 1)
       Snippet: networking! Networking is key to advancing each step of your development and career. 85% of jobs are filled through networking 1 |  COLLECT INFORMATION 2 |  GIVE / RECEIVE ADVICE Aim to find informati...

  [3] Source : d:\career-ai-agent\data\career\Moody_Salary_Negotiation_Guide.pdf  (page 3)
       Snippet: Trust	your	instincts.	Once	you	have	done	your	res

## Observations

| Query | Likely Failure |
|---|---|
| "competencies for ML practitioners" | Vocabulary mismatch (docs say "skills") |
| "salary for Python + ML role" | Multi-hop — salary and skills live in separate chunks |
| "How do I advance?" | Ambiguity — only one intent sampled |

All three failures share one root cause:

```
One Query → One Embedding → One Region Sampled → Top-k chunks
```

The fix: **ask the question multiple ways** — this is MultiQueryRetriever.


---

# Part 2 — MultiQueryRetriever

## Motivation

Single-query retrieval samples one narrow region of the vector space. MultiQueryRetriever paraphrases the question N times, runs each variant through the retriever independently, then merges and deduplicates.

```
Single Query:                     MultiQuery:
─────────────────────────────    ──────────────────────────────────────
"What skills for ML Engineer?"    Original + N paraphrases
         │                              │          │         │
         ▼                              ▼          ▼         ▼
   [One embedding]               [Embed 1]  [Embed 2]  [Embed 3]
         │                              │          │         │
     Top-k only                    Top-k      Top-k     Top-k
                                        └──────────┴─────────┘
                                                  ▼
                                       Union + Deduplicate
                                                  │
                                        Broader, richer context
```


## Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                    MultiQueryRetriever                       │
│                                                              │
│  Input: { question, chat_history }                          │
│               │                                             │
│               ▼                                             │
│    ┌─────────────────────┐                                  │
│    │  Query Generator    │  ChatPromptTemplate + LLM        │
│    └─────────────────────┘                                  │
│          │         │         │                              │
│     [Query 1]  [Query 2]  [Query 3]                        │
│          │         │         │                              │
│      Retriever Retriever Retriever  (base_retriever × N)   │
│          │         │         │                              │
│     [Docs]    [Docs]     [Docs]                            │
│          └─────────┴─────────┘                              │
│                    │                                        │
│    ┌─────────────────────┐                                  │
│    │  Deduplicate by     │  page_content identity           │
│    │  page_content       │                                  │
│    └─────────────────────┘                                  │
│                    │                                        │
│          Unique Retrieved Documents                         │
└──────────────────────────────────────────────────────────────┘
```

| Component | Role |
|---|---|
| Query Generator (LLM) | Paraphrases the original question into N alternatives |
| Base Retriever | Standard similarity search — called once per generated query |
| Deduplication | Merges results by content identity — no duplicate chunks |


## Manual Implementation

We build every internal step explicitly — making the pipeline fully inspectable and debuggable before using the convenience API.

### Step 1 — Query Generator


In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── System prompt instructs the LLM to act as a query paraphrase engine ──────
QUERY_GENERATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are an expert search query optimizer.

Your task is to generate {n_queries} different versions of the given question.

RULES:
1. Each version must preserve the original meaning.
2. Use different vocabulary, phrasing, and structure.
3. Think about synonyms and related terms the document might use.
4. Each query must be on its own line.
5. Output ONLY the queries — no bullets, no numbering, no explanations.

The goal is to improve document retrieval by casting a wider semantic net."""),
    ("human", "Original question: {question}")
])

# ── Query generator: Prompt → LLM → raw string of N lines ───────────────────
query_generator_chain = QUERY_GENERATION_PROMPT | llm | StrOutputParser()

print("✅ Query generator chain built.")


✅ Query generator chain built.


### Step 2 — `generate_queries()`

In [8]:
def generate_queries(question: str, n_queries: int = 3) -> list[str]:
    """
    Use the LLM to paraphrase the original question into n_queries alternatives.

    Args:
        question:  The original user question.
        n_queries: Number of alternative queries to generate.

    Returns:
        A list of query strings (including the original).
    """
    raw_output = query_generator_chain.invoke({
        "question": question,
        "n_queries": n_queries,
    })

    # Split on newlines, strip whitespace, remove empty lines
    generated = [q.strip() for q in raw_output.split("\n") if q.strip()]

    # Always include the original question to guarantee it is covered
    all_queries = [question] + generated

    return all_queries


### Step 3 — `retrieve_for_query()` and `retrieve_all()`

In [9]:
from langchain_core.documents import Document

def retrieve_for_query(query: str, retriever, k: int = 3) -> list[Document]:
    """
    Run a single query through the retriever and return its results.

    Args:
        query:     The query string to search with.
        retriever: A LangChain retriever (any object with .invoke()).
        k:         Max chunks to retrieve (passed via search_kwargs).

    Returns:
        A list of Document objects.
    """
    return retriever.invoke(query)


def retrieve_all(queries: list[str], retriever) -> dict[str, list[Document]]:
    """
    Run every query through the retriever and collect all results.

    Args:
        queries:   List of query strings.
        retriever: A LangChain retriever.

    Returns:
        A dict mapping each query string → its retrieved Document list.
    """
    results = {}
    for query in queries:
        results[query] = retrieve_for_query(query, retriever)
    return results


### Step 4 — `deduplicate_documents()`

In [10]:
def deduplicate_documents(retrieval_map: dict[str, list[Document]]) -> list[Document]:
    """
    Merge all per-query results into a single unique list.

    Identity is determined by the document's page_content string.
    Metadata differences for identical content are ignored — we keep the first occurrence.

    Args:
        retrieval_map: Dict of { query_str -> List[Document] }.

    Returns:
        A deduplicated list of unique Document objects.
    """
    seen_content: set[str] = set()
    unique_docs: list[Document] = []

    for query, docs in retrieval_map.items():
        for doc in docs:
            # Use content as the deduplication key
            content_key = doc.page_content.strip()
            if content_key not in seen_content:
                seen_content.add(content_key)
                unique_docs.append(doc)

    return unique_docs


### Step 5 — `multi_query_retrieve()` — Full Pipeline

In [11]:
def multi_query_retrieve(
    question: str,
    retriever,
    n_queries: int = 3,
    verbose: bool = True
) -> list[Document]:
    """
    Complete Manual MultiQueryRetriever pipeline.

    Steps:
        1. Generate N paraphrased queries from the original question.
        2. Run each query through the retriever.
        3. Merge and deduplicate all results.
        4. (Optional) Print a verbose trace of every intermediate step.

    Args:
        question:  Original user question.
        retriever: A LangChain retriever.
        n_queries: Number of additional paraphrased queries to generate.
        verbose:   If True, print a full trace of the pipeline.

    Returns:
        A deduplicated list of all retrieved Document objects.
    """
    # ── Step 1: Generate queries ──────────────────────────────────────────────
    queries = generate_queries(question, n_queries=n_queries)

    if verbose:
        print(f"{'═'*65}")
        print(f"  ORIGINAL QUESTION")
        print(f"  {question}")
        print(f"{'─'*65}")
        print(f"  GENERATED QUERIES ({len(queries)} total including original)")
        for i, q in enumerate(queries):
            label = "  [original]" if i == 0 else f"  [variant {i}]"
            print(f"  {label}  {q}")
        print(f"{'═'*65}")

    # ── Step 2: Retrieve for every query ─────────────────────────────────────
    retrieval_map = retrieve_all(queries, retriever)

    if verbose:
        print()
        for query, docs in retrieval_map.items():
            print(f"  QUERY: {query[:80]}...")
            print(f"  → {len(docs)} chunk(s) retrieved")
            for j, doc in enumerate(docs, 1):
                src = doc.metadata.get("source", "unknown")
                snippet = doc.page_content.replace("\n", " ").strip()[:120]
                print(f"    [{j}] {src}: {snippet}...")
            print()

    # ── Step 3: Merge and deduplicate ─────────────────────────────────────────
    unique_docs = deduplicate_documents(retrieval_map)

    if verbose:
        total_raw = sum(len(docs) for docs in retrieval_map.values())
        print(f"{'─'*65}")
        print(f"  MERGE SUMMARY")
        print(f"  Raw retrieved (with duplicates) : {total_raw}")
        print(f"  After deduplication             : {len(unique_docs)}")
        print(f"  Duplicates removed              : {total_raw - len(unique_docs)}")
        print(f"{'═'*65}")

    return unique_docs


## End-to-End Demo

Same three queries from Part 1 — now with full verbose pipeline trace.


In [12]:
# ── Test on Query A (vocabulary mismatch) ────────────────────────────────────
question_a = "What competencies are essential for machine learning practitioners?"

print("\n" + "▶" * 5 + "  QUERY A  " + "◀" * 5)
docs_a_multi = multi_query_retrieve(question_a, base_retriever, n_queries=3, verbose=True)



▶▶▶▶▶  QUERY A  ◀◀◀◀◀
═════════════════════════════════════════════════════════════════
  ORIGINAL QUESTION
  What competencies are essential for machine learning practitioners?
─────────────────────────────────────────────────────────────────
  GENERATED QUERIES (4 total including original)
    [original]  What competencies are essential for machine learning practitioners?
    [variant 1]  Which skills are crucial for professionals working in machine learning?
    [variant 2]  What abilities should individuals possess to excel in the field of machine learning?
    [variant 3]  What key qualifications do machine learning experts need to have?
═════════════════════════════════════════════════════════════════

  QUERY: What competencies are essential for machine learning practitioners?...
  → 3 chunk(s) retrieved
    [1] d:\career-ai-agent\data\resumes\ResumeGuideforStudentsGeneral.pdf: The UCC offers several options for getting help in creating and perfecting your resume.  Take advanta

In [13]:
# ── Test on Query B (multi-hop) ───────────────────────────────────────────────
question_b = "What is the salary range for a role that requires Python and machine learning skills?"

print("\n" + "▶" * 5 + "  QUERY B  " + "◀" * 5)
docs_b_multi = multi_query_retrieve(question_b, base_retriever, n_queries=3, verbose=True)



▶▶▶▶▶  QUERY B  ◀◀◀◀◀
═════════════════════════════════════════════════════════════════
  ORIGINAL QUESTION
  What is the salary range for a role that requires Python and machine learning skills?
─────────────────────────────────────────────────────────────────
  GENERATED QUERIES (4 total including original)
    [original]  What is the salary range for a role that requires Python and machine learning skills?
    [variant 1]  What is the compensation range for a position that involves Python and machine learning expertise?
    [variant 2]  How much can one expect to earn in a job that necessitates skills in Python and machine learning?
    [variant 3]  What is the pay scale for a role that demands proficiency in Python and machine learning?
═════════════════════════════════════════════════════════════════

  QUERY: What is the salary range for a role that requires Python and machine learning sk...
  → 3 chunk(s) retrieved
    [1] d:\career-ai-agent\data\resumes\ResumeGuideforStudentsG

In [14]:
# ── Test on Query C (ambiguous) ───────────────────────────────────────────────
question_c = "How do I advance?"

print("\n" + "▶" * 5 + "  QUERY C  " + "◀" * 5)
docs_c_multi = multi_query_retrieve(question_c, base_retriever, n_queries=3, verbose=True)



▶▶▶▶▶  QUERY C  ◀◀◀◀◀
═════════════════════════════════════════════════════════════════
  ORIGINAL QUESTION
  How do I advance?
─────────────────────────────────────────────────────────────────
  GENERATED QUERIES (4 total including original)
    [original]  How do I advance?
    [variant 1]  What steps can I take to move forward?
    [variant 2]  How can I progress in my endeavors?
    [variant 3]  In what ways can I promote my advancement?
═════════════════════════════════════════════════════════════════

  QUERY: How do I advance?...
  → 3 chunk(s) retrieved
    [1] d:\career-ai-agent\data\interview\CMU_Behavioral_Interview_Guide.pdf: Before the Interview:   • Spend time thoroughly researching the company. You will likely be asked why you are  intereste...
    [2] d:\career-ai-agent\data\career\UMich_Alumni_Networking_Guide.pdf: networking! Networking is key to advancing each step of your development and career. 85% of jobs are filled through netw...
    [3] d:\career-ai-agent\data

## Official LangChain API — Reference

`MultiQueryRetriever` is in `langchain_classic` in this environment (standard install: `langchain.retrievers.multi_query`).  
The snippets below are **shown for reference only** — the manual implementation above is the live executable version.


```python
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# ── Build the official MultiQueryRetriever ────────────────────────────────────
# It takes:
#   - retriever: the base similarity search retriever
#   - llm: the language model used to generate query variants
# Internally it builds the same generate → retrieve → merge pipeline we built above
official_multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
)

print("✅ Official MultiQueryRetriever built successfully.")
print(f"   Type: {type(official_multi_retriever)}")

```

Enable internal query logging to see generated variants:

```python
import logging

# Enable LangChain's multi-query logger at INFO level
# This will print the generated queries to the console
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Run the official retriever on Question A
official_docs_a = official_multi_retriever.invoke(question_a)
print(f"\n✅ Official retriever returned {len(official_docs_a)} unique chunk(s) for Query A.")

```

Run on Questions B and C:

```python
# Run on Question B
official_docs_b = official_multi_retriever.invoke(question_b)
print(f"✅ Official retriever returned {len(official_docs_b)} unique chunk(s) for Query B.")

# Run on Question C
official_docs_c = official_multi_retriever.invoke(question_c)
print(f"✅ Official retriever returned {len(official_docs_c)} unique chunk(s) for Query C.")

```

## Side-by-Side Comparison

In [15]:
# ── Run base_retriever on all three questions ─────────────────────────────────
base_docs_a = base_retriever.invoke(question_a)
base_docs_b = base_retriever.invoke(question_b)
base_docs_c = base_retriever.invoke(question_c)

# ── Build comparison table (manual multi-query vs base) ───────────────────────
rows = [
    ("Query A (vocabulary mismatch)",
        len(base_docs_a), len(docs_a_multi)),
    ("Query B (multi-hop)",
        len(base_docs_b), len(docs_b_multi)),
    ("Query C (ambiguous)",
        len(base_docs_c), len(docs_c_multi)),
]

header = f"{'Query':<35} {'Base':>6} {'Manual MultiQuery':>18}"
print(header)
print('─' * len(header))
for name, base, manual in rows:
    print(f"{name:<35} {base:>6} {manual:>18}")

print()
print('* Higher = more unique chunks retrieved (better recall)')
print('* MultiQuery almost always retrieves more unique chunks than Base.')


Query                                 Base  Manual MultiQuery
─────────────────────────────────────────────────────────────
Query A (vocabulary mismatch)            3                  9
Query B (multi-hop)                      3                  7
Query C (ambiguous)                      3                 10

* Higher = more unique chunks retrieved (better recall)
* MultiQuery almost always retrieves more unique chunks than Base.


## Tradeoffs & Production Notes

### Comparison

| Dimension | Base Retriever | MultiQueryRetriever |
|---|---|---|
| LLM calls per query | 0 | 1 (query generation) |
| Retriever calls per query | 1 | N |
| Recall | Lower | Higher |
| Precision | Higher | Slightly lower (broader net) |
| Latency | Fast | Slower (N+1 calls) |
| Cost | Minimal | Higher |
| Vocabulary mismatch | ❌ | ✅ |
| Multi-hop questions | ❌ | ⚠️ Partial |
| Ambiguous queries | Unpredictable | More robust |

### ✅ Use When
- Queries are casual, ambiguous, or domain-specific.
- Missing a chunk causes a meaningfully wrong answer.
- Latency budget allows 500–2000ms extra per turn.

### ❌ Skip When
- Queries are well-formed and precise.
- Latency is critical or traffic is very high.

### Common Mistakes
```
❌ n_queries too high (e.g., 10) → semantic drift, precision drops, latency spikes
❌ Expensive LLM for query gen  → use GPT-4o-mini/Claude Haiku; paraphrase is simple
❌ Skipping deduplication       → duplicate chunks waste context; LLM reads same info twice
❌ Universal drop-in            → profile query types first; multi-query isn't always better
```

### Tuning

| Parameter | Recommended | Notes |
|---|---|---|
| `n_queries` | 2–4 | Beyond 4, marginal gains rarely justify cost |
| LLM for generation | Smaller/faster model | Save your expensive model for the final answer |
| `k` per query | 2–4 | 3 queries × k=3 → up to 9 unique chunks |


## Key Takeaways

```
Naive RAG fails because:
  ├── One query → One embedding → One narrow region sampled
  ├── Vocabulary mismatch, multi-hop, ambiguity all exploit this
  └── Fixed Top-k is never the right k for all questions

MultiQueryRetriever fixes this:
  ├── LLM paraphrases the question N times
  ├── Retriever runs once per paraphrase
  ├── Results merged and deduplicated
  └── Better recall at the cost of N+1 calls and extra latency
```

**Next**: Part 3 — Contextual Compression Retriever  
*(Retrieves broadly, then filters down to only the relevant portion of each chunk.)*


---

# Part 3 — ParentDocumentRetriever


## 3.1 Motivation — The Chunking Dilemma

Every RAG system must split documents into chunks before indexing. The chunk size choice creates an unavoidable tradeoff:

| Chunk Size | Retrieval Precision | Context Quality | Problem |
|---|---|---|---|
| **Small** (100–200 tokens) | ✅ High — exact match | ❌ Low — fragment loses surrounding meaning | LLM gets a sentence with no context |
| **Large** (1000–2000 tokens) | ❌ Low — diluted embedding | ✅ High — full context preserved | Wrong chunk wins; irrelevant text crowds the answer |

**There is no perfect chunk size** for both retrieval and generation simultaneously.

```
Small Chunk Problem:
─────────────────────────────────────────────────────────
Original paragraph:
  "The Senior ML Engineer role requires deep expertise in
   distributed training frameworks. Candidates must have
   3+ years of PyTorch experience and familiarity with
   model parallelism strategies."

After small chunking (chunk_size=100):
  Chunk A: "The Senior ML Engineer role requires deep"
  Chunk B: "expertise in distributed training frameworks."
  Chunk C: "Candidates must have 3+ years of PyTorch"
  Chunk D: "experience and familiarity with model"
  Chunk E: "parallelism strategies."

Query: "What PyTorch experience is needed?"
Retrieved: Chunk C — "Candidates must have 3+ years of PyTorch"
                      ↑ Misses: what role? what for? no context.
─────────────────────────────────────────────────────────

Large Chunk Problem:
─────────────────────────────────────────────────────────
After large chunking (chunk_size=2000):
  One chunk contains: job title + requirements +
  salary + benefits + company culture + team size

Query: "What PyTorch experience is needed?"
Retrieved: The entire 2000-token blob
           ↑ LLM must read 2000 tokens to find one sentence.
           ↑ Embedding diluted — less precise retrieval.
─────────────────────────────────────────────────────────
```

**The solution:** use small chunks for retrieval precision, but return the full parent document for context quality. This is the insight behind `ParentDocumentRetriever`.


## 3.2 Intuition — Search Small, Return Big

`ParentDocumentRetriever` separates the retrieval unit from the generation unit:

```
INDEXING TIME:
──────────────────────────────────────────────────────────────
  Full Document
       │
       ├─► Parent Chunks (large, ~500–2000 tokens)
       │     stored in DocStore (InMemoryStore / Redis)
       │
       └─► Child Chunks (small, ~100–200 tokens)
             embedded and stored in Vector Store

RETRIEVAL TIME:
──────────────────────────────────────────────────────────────
  User Query
       │
       ▼
  Embed Query
       │
       ▼
  Similarity Search against CHILD chunk embeddings
  (small → precise match)
       │
       ▼
  Retrieve matched Child Chunk
       │
       ▼
  Look up Parent ID from Child metadata
       │
       ▼
  Fetch PARENT Chunk from DocStore
  (large → rich context)
       │
       ▼
  Return Parent Chunk to LLM
```

**Why this works:**  
- The small child chunk gives the embedding a tight, focused semantic signal → high precision.  
- The large parent document gives the LLM the surrounding context it needs → high quality answers.


## 3.3 Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                    ParentDocumentRetriever                          │
│                                                                     │
│  INDEXING PIPELINE                                                  │
│  ─────────────────                                                  │
│  Raw Documents                                                      │
│       │                                                             │
│       ├──[parent_splitter]──► Parent Chunks ──► InMemoryStore       │
│       │                         (large)          (key-value store)  │
│       │                                               ↑             │
│       └──[child_splitter]───► Child Chunks ────► Vector Store       │
│                                 (small)          (Chroma)           │
│                               + parent_id                           │
│                                 in metadata                         │
│                                                                     │
│  RETRIEVAL PIPELINE                                                 │
│  ──────────────────                                                 │
│  User Query                                                         │
│       │ embed                                                       │
│       ▼                                                             │
│  Vector Store (Child) ──► Top-k Child Chunks                        │
│       │                         │                                   │
│       │                    read parent_id                           │
│       │                         │                                   │
│       ▼                         ▼                                   │
│  InMemoryStore ─────────► Parent Documents                         │
│  (lookup by parent_id)          │                                   │
│                                 ▼                                   │
│                           Returned to LLM                          │
└─────────────────────────────────────────────────────────────────────┘
```

| Component | Role |
|---|---|
| `child_splitter` | Splits documents into small chunks for embedding (precision) |
| `parent_splitter` | Splits documents into large chunks for the LLM (context quality) |
| `Vector Store` | Stores child chunk embeddings — used for similarity search |
| `InMemoryStore` | Key-value store mapping `parent_id → parent document` |


## 3.4 Traditional Chunking Demo

Before implementing `ParentDocumentRetriever`, let's observe the context problem with the existing `base_retriever` (which was built on standard small chunks).

We'll retrieve chunks and inspect the context loss firsthand.


In [16]:
# ── Inspect existing chunk sizes in the vector store ─────────────────────────
raw = vector_db.get(limit=5, include=["documents", "metadatas"])

print("=== Sample chunks from existing Vector Store ===")
for i, (doc, meta) in enumerate(zip(raw["documents"], raw["metadatas"])):
    source = meta.get("source", "unknown")
    print(f"\n[Chunk {i+1}] Source: {source}")
    print(f"  Length : {len(doc)} characters (~{len(doc.split())} words)")
    print(f"  Content: {doc[:200].strip()}...")


=== Sample chunks from existing Vector Store ===

[Chunk 1] Source: d:\career-ai-agent\data\interview\CMU_Behavioral_Interview_Guide.pdf
  Length : 953 characters (~153 words)
  Content: Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or more behavioral interviews 
with a company of interest. If you are ever unsure of the type of inter...

[Chunk 2] Source: d:\career-ai-agent\data\interview\CMU_Behavioral_Interview_Guide.pdf
  Length : 962 characters (~152 words)
  Content: Before the Interview:  
• Spend time thoroughly researching the company. You will likely be asked why you are 
interested in the company, and you want to be able to connect your interests back 
to the...

[Chunk 3] Source: d:\career-ai-agent\data\interview\CMU_Behavioral_Interview_Guide.pdf
  Length : 947 characters (~158 words)
  Content: with. These questions should be authentic to your curiosities and should not be 
easily answered by searching the internet or the

Now run a query through `base_retriever` and observe what gets returned — notice how the retrieved chunk is a fragment that lacks surrounding context.


In [17]:
# ── Demo query: observe fragment retrieval with base_retriever ───────────────
demo_query = "What Python experience is required for the ML Engineer role?"

print(f"Query: {demo_query}\n")
base_chunks = base_retriever.invoke(demo_query)

for i, chunk in enumerate(base_chunks, 1):
    source = chunk.metadata.get("source", "unknown")
    page   = chunk.metadata.get("page", "N/A")
    print(f"[Retrieved Chunk {i}]")
    print(f"  Source : {source} (page {page})")
    print(f"  Length : {len(chunk.page_content)} chars")
    print(f"  Content:\n  {chunk.page_content.strip()}")
    print()

print("─" * 60)
print("Observation: Each chunk is a fragment.")
print("The LLM receives isolated sentences without surrounding context.")


Query: What Python experience is required for the ML Engineer role?

[Retrieved Chunk 1]
  Source : d:\career-ai-agent\data\interview\10-41-Essential-Machine-Learning-Interview-Questions.pdf (page 18)
  Length : 973 chars
  Content:
  This kind of question requires you to listen carefully and impart feed-
back in a manner that is constructive and insightful. Your interviewer 
is trying to gauge if you’d be a valuable member of their team and 
whether you grasp the nuances of why certain things are set the way 
they are in the company’s data process based on company- or indus-
try-specific conditions. They’re trying to see if you can be an intellec-
tual peer. Act accordingly.
Machine Learning Interview Questions: General 
Machine Learning Interest
This series of machine learning interview questions attempts to gauge 
your passion and interest in machine learning. The right answers will 
serve as a testament for your commitment to being a lifelong learner 
in machine learning.
Q35- What

## 3.5 ParentDocumentRetriever Implementation

We build a `ParentDocumentRetriever` using:
- **`child_splitter`** — small chunks (200 tokens) for precision embedding
- **`parent_splitter`** — large chunks (800 tokens) for context-rich generation
- **`InMemoryStore`** — key-value store for parent document lookup
- **`Chroma`** — a *new* Chroma collection dedicated to child chunks only

> **Note:** We create a separate in-memory Chroma collection for child chunks  
> to avoid mixing them with the existing `vector_db`. No re-ingestion of the  
> original source files is needed — we extract documents directly from `vector_db`.


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_core.documents import Document

# ── 1. Retrieve all raw documents from the existing vector store ──────────────
# We extract the indexed content and rebuild Document objects.
# This avoids re-reading PDF files from disk.
raw_data = vector_db.get(include=["documents", "metadatas"])
source_docs = [
    Document(page_content=doc, metadata=meta)
    for doc, meta in zip(raw_data["documents"], raw_data["metadatas"])
]
print(f"Extracted {len(source_docs)} chunks from existing Vector DB.")

# ── 2. Define splitters ───────────────────────────────────────────────────────
# Child splitter: small chunks for high-precision embedding
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
)

# Parent splitter: larger chunks that preserve surrounding context
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
)

# ── 3. Create a dedicated in-memory Chroma collection for child chunks ────────
child_vectorstore = Chroma(
    collection_name="parent_doc_children",
    embedding_function=embedding_model,
)

# ── 4. Create the key-value store for parent documents ───────────────────────
parent_store = InMemoryStore()

# ── 5. Build the ParentDocumentRetriever ─────────────────────────────────────
parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=child_vectorstore,       # where child chunk embeddings live
    docstore=parent_store,               # where parent documents are stored
    child_splitter=child_splitter,       # splits parents → children for embedding
    parent_splitter=parent_splitter,     # splits raw docs → parents for storage
)

print("✅ ParentDocumentRetriever built.")


Extracted 285 chunks from existing Vector DB.
✅ ParentDocumentRetriever built.


### Index Documents into the ParentDocumentRetriever

We add our extracted documents. The retriever will automatically:
1. Split them into parent chunks → store in `InMemoryStore`
2. Split parents into child chunks → embed and store in `child_vectorstore`


In [19]:
# ── Add documents to the retriever ───────────────────────────────────────────
# This call splits, embeds, and stores both parents and children automatically.
parent_doc_retriever.add_documents(source_docs, ids=None)

# Verify what was created
child_count  = child_vectorstore._collection.count()
parent_count = len(list(parent_store.yield_keys()))
print(f"Child chunks indexed  : {child_count}")
print(f"Parent documents stored: {parent_count}")
print()
print("Ratio:", round(child_count / max(parent_count, 1), 1), "child chunks per parent")


Child chunks indexed  : 1701
Parent documents stored: 497

Ratio: 3.4 child chunks per parent


## 3.6 Debugging — Tracing Every Retrieval Step

A key advantage of building this manually is full observability. Let's trace exactly what happens for each query step-by-step.


In [20]:
# ── Step-by-step trace of ParentDocumentRetriever internals ──────────────────

def trace_parent_retrieval(query: str, retriever: ParentDocumentRetriever) -> list[Document]:
    """
    Manually trace the full retrieval pipeline:
      1. Embed query → find matching child chunks
      2. Extract parent_id from child metadata
      3. Fetch parent documents from docstore
      4. Return parent documents
    """
    print(f"{'═'*65}")
    print(f"  QUERY: {query}")
    print(f"{'─'*65}")

    # Step 1: Search child chunk vector store directly
    child_results = retriever.vectorstore.similarity_search(query, k=4)
    print(f"\n[Step 1] Child chunks matched: {len(child_results)}")
    for i, child in enumerate(child_results, 1):
        parent_id = child.metadata.get("doc_id", child.metadata.get("parent_id", "unknown"))
        print(f"  Child {i}: len={len(child.page_content)} chars | parent_id={parent_id}")
        print(f"    Snippet: {child.page_content[:100].strip()}...")

    # Step 2: Collect unique parent IDs
    parent_ids = list({
        child.metadata.get("doc_id", child.metadata.get("parent_id", ""))
        for child in child_results
        if child.metadata.get("doc_id") or child.metadata.get("parent_id")
    })
    print(f"\n[Step 2] Unique parent IDs: {parent_ids}")

    # Step 3: Fetch parents from docstore
    if parent_ids:
        parents = list(retriever.docstore.mget(parent_ids))
        parents = [p for p in parents if p is not None]
    else:
        parents = []

    print(f"\n[Step 3] Parent documents fetched: {len(parents)}")
    for i, parent in enumerate(parents, 1):
        print(f"  Parent {i}: len={len(parent.page_content)} chars")
        print(f"    Content: {parent.page_content[:150].strip()}...")

    print(f"\n[Step 4] Final context passed to LLM: {sum(len(p.page_content) for p in parents)} total chars")
    print(f"{'═'*65}\n")
    return parents


debug_query = "What Python experience is required for the ML Engineer role?"
traced_docs = trace_parent_retrieval(debug_query, parent_doc_retriever)


═════════════════════════════════════════════════════════════════
  QUERY: What Python experience is required for the ML Engineer role?
─────────────────────────────────────────────────────────────────

[Step 1] Child chunks matched: 4
  Child 1: len=101 chars | parent_id=a08c4309-db75-4639-9101-771635e7b4d4
    Snippet: Experience, Honors, Activities, Technical Skills, Relevant Interests, and Professional Development....
  Child 2: len=171 chars | parent_id=63979d49-9d86-460d-8a84-5448a4c3dfab
    Snippet: SKILLS  
Programming: Python (numpy, pandas, scikit-learn, pytorch), SQL, R, Bloomberg Terminal, MAT...
  Child 3: len=152 chars | parent_id=f75f704d-a37c-4227-9fa2-e48fb4f8095b
    Snippet: • Promoted to Field Application Engineer
• Hired after completing challenging summer internship, qui...
  Child 4: len=166 chars | parent_id=138ba7bf-931f-4360-970c-b933ca198bcd
    Snippet: between your past experiences, skills and overall qualifications and the position 
requirements. Hig...



Now invoke the retriever via the standard `.invoke()` interface and compare:


In [21]:
# ── Standard invocation (same result, no trace) ───────────────────────────────
retrieved_parents = parent_doc_retriever.invoke(demo_query)

print(f"ParentDocumentRetriever returned {len(retrieved_parents)} parent document(s).\n")
for i, doc in enumerate(retrieved_parents, 1):
    source = doc.metadata.get("source", "unknown")
    print(f"[Parent {i}] Source: {source} | Length: {len(doc.page_content)} chars")
    print(f"  Content:\n  {doc.page_content.strip()[:400]}...")
    print()


ParentDocumentRetriever returned 4 parent document(s).

[Parent 1] Source: d:\career-ai-agent\data\resumes\ResumeGuideforStudentsGeneral.pdf | Length: 784 chars
  Content:
  med student shadowing at a hospital).  
 
Example 1: 
Observation, University Medical Center, Lubbock, Texas     October 2024 
 
Example 2: 
Band Library Clean Up, Tau Beta Sigma, Lubbock, Texas           March 2023 
 
Other Work or Professional Experience 
Employers and the selection committee will be most interested in your relevant experience; however, you 
may include skills obtained through o...

[Parent 2] Source: d:\career-ai-agent\data\resumes\MASTERS-RESUME-GUIDE.pdf | Length: 728 chars
  Content:
  SKILLS  
Programming: Python (numpy, pandas, scikit-learn, pytorch), SQL, R, Bloomberg Terminal, MATLAB, Latex 
Language: Fluent in Korean and Chinese 
 
RELEVANT RESEARCH  
Harvard University,                                                                                                                       

## 3.7 Comparison — Three Retrievers Side by Side

We run all three retrieval strategies on the same set of questions and compare context quality, recall, and token cost.


In [22]:
from langchain_core.messages import HumanMessage, AIMessage

# ── Test questions ─────────────────────────────────────────────────────────────
test_questions = [
    "What Python experience is required for the ML Engineer role?",
    "What are the career advancement opportunities in data science?",
    "How do I transition from software engineering to machine learning?",
]

# ── Run all three retrievers on each question ──────────────────────────────────
print("=" * 70)
print(f"{'Question':<45} {'Base':>5} {'MultiQ':>7} {'ParentDoc':>10}")
print(f"{'':45} {'chunks':>5} {'chunks':>7} {'chunks':>10}")
print("─" * 70)

for q in test_questions:
    base_docs   = base_retriever.invoke(q)
    # multi_query: reuse function from Part 2
    multi_docs  = multi_query_retrieve(q, base_retriever, n_queries=3, verbose=False)
    parent_docs = parent_doc_retriever.invoke(q)

    q_short = q[:44] + "…" if len(q) > 45 else q
    print(f"{q_short:<45} {len(base_docs):>5} {len(multi_docs):>7} {len(parent_docs):>10}")

print("=" * 70)


Question                                       Base  MultiQ  ParentDoc
                                              chunks  chunks     chunks
──────────────────────────────────────────────────────────────────────
What Python experience is required for the M…     3       7          4
What are the career advancement opportunitie…     3      10          4
How do I transition from software engineerin…     3       7          4


In [23]:
# ── Context quality comparison: token count per strategy ──────────────────────
print("\n=== Context Size (characters) per Retriever ===\n")

for q in test_questions[:2]:
    base_docs   = base_retriever.invoke(q)
    multi_docs  = multi_query_retrieve(q, base_retriever, n_queries=3, verbose=False)
    parent_docs = parent_doc_retriever.invoke(q)

    base_chars   = sum(len(d.page_content) for d in base_docs)
    multi_chars  = sum(len(d.page_content) for d in multi_docs)
    parent_chars = sum(len(d.page_content) for d in parent_docs)

    print(f"Q: {q[:60]}")
    print(f"  Base Retriever       : {base_chars:>6} chars")
    print(f"  MultiQueryRetriever  : {multi_chars:>6} chars")
    print(f"  ParentDocRetriever   : {parent_chars:>6} chars")
    print()



=== Context Size (characters) per Retriever ===

Q: What Python experience is required for the ML Engineer role?
  Base Retriever       :   2863 chars
  MultiQueryRetriever  :   7089 chars
  ParentDocRetriever   :   2385 chars

Q: What are the career advancement opportunities in data scienc
  Base Retriever       :   2840 chars
  MultiQueryRetriever  :   5681 chars
  ParentDocRetriever   :   2036 chars



### Retriever Comparison Table

| Dimension | Base Retriever | MultiQueryRetriever | ParentDocumentRetriever |
|---|---|---|---|
| **Retrieval Unit** | Small chunk | Small chunk × N queries | Child chunk → Parent returned |
| **Context Quality** | ❌ Fragment only | ❌ Fragments (more of them) | ✅ Full surrounding context |
| **Recall** | Low | Higher | Medium-High |
| **Precision** | High | Slightly lower | High (child is precise) |
| **LLM Calls** | 0 | 1 (query gen) | 0 |
| **Retriever Calls** | 1 | N | 1 (child) + N lookups |
| **Memory (RAM)** | Low | Low | Medium (parent DocStore in RAM) |
| **Token Usage** | Low | Medium | Higher (parents are larger) |
| **Latency** | Fastest | Slow (N+1 LLM) | Fast (no extra LLM call) |
| **Vocabulary Mismatch** | ❌ | ✅ | ❌ (still single query) |
| **Context Fragmentation** | ❌ | ❌ | ✅ |
| **Best For** | Simple, precise queries | Ambiguous/multi-intent queries | Context-dependent answers |


## 3.8 Production Notes & Best Practices

### ✅ Use ParentDocumentRetriever When
- Answers require surrounding context (e.g., procedural steps, comparisons, tables).
- Your documents have rich structure that gets destroyed by small chunking.
- Users ask follow-up questions that depend on what came before in the source text.
- You're indexing long-form content: research papers, manuals, legal docs.

### ❌ Avoid When
- Your vector store already holds well-sized, context-complete chunks.
- RAM is constrained — `InMemoryStore` holds all parent docs in RAM.
- You need persistence across restarts — `InMemoryStore` is not persistent.
- Documents are already short (FAQ entries, single-sentence facts).

### Common Mistakes

```
❌ child_size ≥ parent_size
   → Parents must always be larger than children.
   → Rule: child_size × 4–6 ≈ parent_size (e.g., 200 child / 800 parent)

❌ Using InMemoryStore in production at scale
   → InMemoryStore holds all parents in RAM — OOM risk at large scale.
   → Production: use RedisStore, MongoDBStore, or DynamoDBStore.

❌ Mixing child and parent chunks in the same vector store
   → Parent chunk embeddings pollute the child search space.
   → Always use a separate collection for child embeddings.

❌ No parent_splitter (using full documents as parents)
   → Full documents as parents means every retrieval returns thousands of tokens.
   → Context window explodes. Always define a parent_splitter.
```

### Chunk Size Recommendations

| Document Type | Child Size | Parent Size | Overlap |
|---|---|---|---|
| Technical docs / manuals | 150–200 tokens | 600–900 tokens | 10–15% |
| Research papers | 200–300 tokens | 800–1200 tokens | 10% |
| Legal / financial docs | 100–150 tokens | 500–800 tokens | 15–20% |
| FAQ / knowledge bases | 50–100 tokens | 200–400 tokens | 5–10% |

### Memory & Persistence

| DocStore | Persistence | Scale | Notes |
|---|---|---|---|
| `InMemoryStore` | ❌ Lost on restart | Small datasets | Use for prototyping only |
| `RedisStore` | ✅ Persistent | Large scale | Production-ready |
| `MongoDBStore` | ✅ Persistent | Large scale | Good for structured metadata |
| `LocalFileStore` | ✅ On-disk | Medium scale | Simple persistence without a DB |


## 3.9 Engineering Best Practices

- **Profile before optimizing**: Run your queries against `base_retriever` first. If answers are already good, `ParentDocumentRetriever` adds cost without benefit.
- **Separate collections**: Always index child chunks in a dedicated vector store collection, never mixed with your main store.
- **Tune chunk sizes experimentally**: Start with child=200, parent=800. Measure answer quality. Adjust in increments of 100 tokens.
- **Combine with MultiQuery**: `ParentDocumentRetriever` solves context fragmentation. `MultiQueryRetriever` solves vocabulary mismatch. They compose — use `ParentDocumentRetriever` as the base retriever inside `MultiQueryRetriever` for both benefits.
- **Production DocStore**: Replace `InMemoryStore` with a persistent store (`RedisStore`, `MongoDBStore`) before deploying. Never rely on in-memory state across server restarts.
- **Monitor token costs**: Parent documents are larger — track your average tokens-per-query in production. Use a `ContextualCompressionRetriever` downstream if parents are too verbose.


## 3.10 Summary

```
The Chunking Dilemma:
  Small chunks → high precision retrieval, poor context for LLM
  Large chunks → poor precision retrieval, good context for LLM
  No single chunk size solves both simultaneously.

ParentDocumentRetriever solution:
  Index SMALL child chunks for embedding (precision)
  Store LARGE parent chunks in a docstore (context quality)
  Retrieve: child chunk matched → parent document returned to LLM

Key tradeoffs:
  ✅ Best context quality of all three retrievers
  ✅ No extra LLM call at retrieval time (unlike MultiQuery)
  ❌ Higher RAM usage (parent DocStore lives in memory)
  ❌ Higher token cost per query (parents are larger)
  ❌ InMemoryStore not persistent — needs swap in production
```

---

**Retriever Decision Guide:**

| Scenario | Recommended Retriever |
|---|---|
| Simple, well-formed queries | `base_retriever` |
| Ambiguous / multi-intent queries | `MultiQueryRetriever` |
| Context-fragmented answers | `ParentDocumentRetriever` |
| Both ambiguity + fragmentation | `MultiQueryRetriever(base=ParentDocRetriever)` |

---

**Next**: Part 4 — Contextual Compression Retriever  
*(Retrieves broadly, then filters down to only the portions of each document relevant to the query.)*


---

# Part 4 — Contextual Compression Retriever


## 4.1 Why Contextual Compression?

In Part 3, `ParentDocumentRetriever` solved context fragmentation by returning large parent chunks.  
But this created a new problem: **the LLM now receives entire paragraphs when it only needs one sentence.**

```
Query: "What Python version is required?"

ParentDocumentRetriever returns (800 tokens):
  "Our engineering team is organized across three offices globally.
   We use agile methodologies and have biweekly sprints. The role
   requires a Bachelor's degree in Computer Science or equivalent.
   Candidates must have Python 3.9+ experience and knowledge of
   async programming. We offer competitive salaries and flexible
   remote work options. The team uses GitHub for version control..."

LLM actually needs (12 tokens):
  "Candidates must have Python 3.9+ experience."
```

**The cost of sending 800 tokens when 12 suffice:**

| Problem | Impact |
|---|---|
| Wasted tokens | Directly multiplies API cost per query |
| Context dilution | Irrelevant text reduces LLM focus on the answer |
| "Lost in the Middle" | LLM misses the key sentence buried in noise |
| Latency | More tokens → slower first token response |
| Context window pressure | Leaves less space for chat history and system prompt |

The fix is a post-retrieval filtering step that extracts only the passages relevant to the query.  
This is **Contextual Compression**.


## 4.2 The Core Idea

Contextual Compression is a **post-retrieval filter** — not a replacement for the retriever.

> **Important distinction:**  
> Compression ≠ Summarization.  
> The compressor does **not** rephrase or condense text.  
> It **extracts** only the passages from the original document that are relevant to the query,  
> preserving the original wording exactly.

```
WITHOUT Compression:
─────────────────────────────────────────────────────────────
Query ──► Retriever ──► [800-token document] ──► LLM
                         (full parent chunk)

WITH Contextual Compression:
─────────────────────────────────────────────────────────────
Query ──► Retriever ──► [800-token document]
                                │
                                ▼
                         Compressor reads document + query
                                │
                                ▼
                         Extracts only relevant passages
                                │
                                ▼
                         [42-token extract] ──► LLM
```

**Two compression strategies in LangChain:**

| Strategy | How it works | LLM call? | Speed |
|---|---|---|---|
| `LLMChainExtractor` | LLM reads the document and extracts relevant sentences verbatim | ✅ Yes | Slow |
| `LLMChainFilter` | LLM decides YES/NO — keep or discard the whole document | ✅ Yes | Medium |
| `EmbeddingsFilter` | Computes cosine similarity of document to query; drops below threshold | ❌ No | Fast |

We will implement all three.


## 4.3 Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                  ContextualCompressionRetriever                     │
│                                                                     │
│  Input: { query: str }                                              │
│         │                                                           │
│         ▼                                                           │
│  ┌─────────────────┐                                                │
│  │  Base Retriever │  (any retriever: base, MultiQuery, Parent)    │
│  └─────────────────┘                                                │
│         │                                                           │
│         ▼                                                           │
│  Retrieved Documents  [doc_1, doc_2, doc_3]  (full size)           │
│         │                                                           │
│         │  for each document:                                       │
│         ▼                                                           │
│  ┌─────────────────────┐                                            │
│  │  Document Compressor│  (LLMChainExtractor / EmbeddingsFilter)   │
│  │  compress(doc, query│                                            │
│  └─────────────────────┘                                            │
│         │                                                           │
│         ▼                                                           │
│  Compressed Documents  [extract_1, extract_2]  (relevant only)     │
│         │  (some docs may be dropped entirely if irrelevant)        │
│         ▼                                                           │
│                    Returned to LLM                                  │
└─────────────────────────────────────────────────────────────────────┘
```

| Stage | Component | Role |
|---|---|---|
| Retrieval | `base_retriever` | Fetches candidate documents by similarity |
| Compression | `LLMChainExtractor` | Reads each doc, extracts verbatim relevant passages |
| Compression (fast) | `EmbeddingsFilter` | Drops docs below a cosine similarity threshold |
| Output | Compressed docs | Only relevant text, original wording preserved |


## 4.4 Traditional Retrieval Demo — Observing the Noise Problem

Let's measure exactly how much text `base_retriever` sends to the LLM and  
how much of it is actually relevant to the query.


In [24]:
# ── Helper: estimate token count (rough approximation) ───────────────────────
def estimate_tokens(char_count: int) -> int:
    """Rough token estimation: 1 token ≈ 4 characters (GPT-style tokenization)."""
    return char_count // 4


# ── Run a targeted query and inspect what the LLM would receive ───────────────
demo_query_4 = "What Python version and programming skills are required for the ML Engineer role?"

retrieved_docs = base_retriever.invoke(demo_query_4)

total_chars  = sum(len(d.page_content) for d in retrieved_docs)
total_tokens = estimate_tokens(total_chars)

print(f"Query: {demo_query_4}\n")
print(f"{'─'*60}")
print(f"  Documents retrieved : {len(retrieved_docs)}")
print(f"  Total characters    : {total_chars}")
print(f"  Estimated tokens    : ~{total_tokens} tokens")
print(f"{'─'*60}\n")

for i, doc in enumerate(retrieved_docs, 1):
    src    = doc.metadata.get('source', 'unknown')
    chars  = len(doc.page_content)
    tokens = estimate_tokens(chars)
    print(f"[Doc {i}] {src}")
    print(f"  Length  : {chars} chars (~{tokens} tokens)")
    print(f"  Content : {doc.page_content.strip()}")
    print()


Query: What Python version and programming skills are required for the ML Engineer role?

────────────────────────────────────────────────────────────
  Documents retrieved : 3
  Total characters    : 2117
  Estimated tokens    : ~529 tokens
────────────────────────────────────────────────────────────

[Doc 1] d:\career-ai-agent\data\resumes\MASTERS-RESUME-GUIDE.pdf
  Length  : 886 chars (~221 tokens)
  Content : roles in tech as well as generalist positions in big consulting firms. You will want to target each resume to the specific 
employer. For example, employers in tech will be interested in relevant projects and experience analyzing datasets. Be clear 
about how you developed those skills in courses or internship experiences. Consulting firms will be concerned with 
leadership and teamwork skills; in that case, you might want to include more information about your involvement with 
student groups, volunteer work, or internships. Resume writing can be difficult. MCS supports the r

Observe how the retrieved chunks contain sentences unrelated to the query.  
The LLM must read every word before generating an answer — this is the noise problem.


## 4.5 Manual Compression Demo

Before using LangChain's built-in compressors, let's implement a naive sentence-level  
extractor manually. This reveals the exact logic `LLMChainExtractor` automates.


In [25]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Manual extractor: ask the LLM to extract only the relevant sentences ──────
EXTRACT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are a precise text extraction assistant.

Given a document and a question, extract ONLY the sentences from the document
that directly answer or are directly relevant to the question.

STRICT RULES:
1. Copy sentences VERBATIM — do not rephrase or summarize.
2. If no sentence is relevant, respond with exactly: NO_OUTPUT
3. Do not add any explanation, headers, or commentary.
4. Preserve the original language and punctuation."""),
    ("human",
     "QUESTION: {question}\n\nDOCUMENT:\n{document}\n\nExtract only the relevant sentences:"),
])

extract_chain = EXTRACT_PROMPT | llm | StrOutputParser()


def manual_compress(docs: list[Document], query: str) -> list[Document]:
    """
    Manually compress a list of documents by extracting query-relevant sentences.
    Returns a new list of Document objects containing only the extracted passages.
    """
    compressed = []
    for doc in docs:
        result = extract_chain.invoke({
            "question": query,
            "document": doc.page_content,
        })
        # Drop documents where the LLM found nothing relevant
        if result.strip() and result.strip() != "NO_OUTPUT":
            compressed.append(Document(
                page_content=result.strip(),
                metadata={**doc.metadata, "compression": "manual_llm_extractor"},
            ))
    return compressed


# ── Run manual compression ────────────────────────────────────────────────────
print(f"Running manual compression on {len(retrieved_docs)} documents...\n")
manually_compressed = manual_compress(retrieved_docs, demo_query_4)

# ── Before vs After ──────────────────────────────────────────────────────────
original_tokens    = estimate_tokens(sum(len(d.page_content) for d in retrieved_docs))
compressed_tokens  = estimate_tokens(sum(len(d.page_content) for d in manually_compressed))
reduction_pct      = (1 - compressed_tokens / max(original_tokens, 1)) * 100

print(f"{'═'*60}")
print(f"  BEFORE compression: ~{original_tokens} tokens across {len(retrieved_docs)} docs")
print(f"  AFTER compression:  ~{compressed_tokens} tokens across {len(manually_compressed)} docs")
print(f"  Token reduction:    {reduction_pct:.1f}%")
print(f"{'═'*60}\n")

for i, doc in enumerate(manually_compressed, 1):
    print(f"[Extracted {i}]")
    print(f"  {doc.page_content.strip()}")
    print()


Running manual compression on 3 documents...

════════════════════════════════════════════════════════════
  BEFORE compression: ~529 tokens across 3 docs
  AFTER compression:  ~0 tokens across 0 docs
  Token reduction:    100.0%
════════════════════════════════════════════════════════════



## 4.6 Official Implementation — ContextualCompressionRetriever

Now we use LangChain's built-in compressors. We implement three variants:

### Strategy A — `LLMChainExtractor` (highest quality, extra LLM call)


In [26]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# ── Build the extractor compressor ────────────────────────────────────────────
# LLMChainExtractor asks the LLM to extract relevant sentences verbatim
llm_extractor = LLMChainExtractor.from_llm(llm)

# ── Wrap base_retriever with compression ─────────────────────────────────────
extractor_retriever = ContextualCompressionRetriever(
    base_compressor=llm_extractor,
    base_retriever=base_retriever,
)

print("✅ LLMChainExtractor-based ContextualCompressionRetriever built.")


✅ LLMChainExtractor-based ContextualCompressionRetriever built.


### Strategy B — `LLMChainFilter` (keep/discard whole documents)

In [27]:
from langchain_classic.retrievers.document_compressors import LLMChainFilter

# ── Build the filter compressor ───────────────────────────────────────────────
# LLMChainFilter asks the LLM: "Is this document relevant? YES / NO"
# Keeps the whole document if YES, discards entirely if NO
llm_filter = LLMChainFilter.from_llm(llm)

filter_retriever = ContextualCompressionRetriever(
    base_compressor=llm_filter,
    base_retriever=base_retriever,
)

print("✅ LLMChainFilter-based ContextualCompressionRetriever built.")


✅ LLMChainFilter-based ContextualCompressionRetriever built.


### Strategy C — `EmbeddingsFilter` (no LLM call, fastest)

In [28]:
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

# ── Build the embeddings-based filter ────────────────────────────────────────
# EmbeddingsFilter computes cosine similarity between query and each document.
# Documents below the similarity_threshold are dropped — no LLM call needed.
embeddings_filter = EmbeddingsFilter(
    embeddings=embedding_model,
    similarity_threshold=0.76,   # tune: higher = stricter filtering
)

embedding_filter_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever,
)

print("✅ EmbeddingsFilter-based ContextualCompressionRetriever built.")
print("   No extra LLM call — filtering is done via cosine similarity.")


✅ EmbeddingsFilter-based ContextualCompressionRetriever built.
   No extra LLM call — filtering is done via cosine similarity.


## 4.7 Before vs After — Visualizing Compression

Let's run the `extractor_retriever` on our demo query and clearly show  
what was kept, what was removed, and the final context the LLM receives.


In [29]:
# ── Run extractor_retriever and compare to base_retriever output ──────────────
print(f"Query: {demo_query_4}\n")

# Original (uncompressed)
original_docs      = base_retriever.invoke(demo_query_4)
# Compressed (LLMChainExtractor)
compressed_docs    = extractor_retriever.invoke(demo_query_4)

print("=" * 65)
print("  ORIGINAL RETRIEVED DOCUMENTS (base_retriever)")
print("=" * 65)
for i, doc in enumerate(original_docs, 1):
    print(f"  [Doc {i}] {len(doc.page_content)} chars | ~{estimate_tokens(len(doc.page_content))} tokens")
    print(f"  {doc.page_content.strip()[:300]}...")
    print()

print("=" * 65)
print("  AFTER COMPRESSION (LLMChainExtractor)")
print("=" * 65)
for i, doc in enumerate(compressed_docs, 1):
    print(f"  [Extracted {i}] {len(doc.page_content)} chars | ~{estimate_tokens(len(doc.page_content))} tokens")
    print(f"  {doc.page_content.strip()}")
    print()

# ── Summary ───────────────────────────────────────────────────────────────────
orig_tokens = estimate_tokens(sum(len(d.page_content) for d in original_docs))
comp_tokens = estimate_tokens(sum(len(d.page_content) for d in compressed_docs))
print("─" * 65)
print(f"  Docs before compression : {len(original_docs)}")
print(f"  Docs after compression  : {len(compressed_docs)}")
print(f"  Tokens before           : ~{orig_tokens}")
print(f"  Tokens after            : ~{comp_tokens}")
print(f"  Reduction               : {(1 - comp_tokens/max(orig_tokens,1))*100:.1f}%")
print("─" * 65)


Query: What Python version and programming skills are required for the ML Engineer role?

  ORIGINAL RETRIEVED DOCUMENTS (base_retriever)
  [Doc 1] 886 chars | ~221 tokens
  roles in tech as well as generalist positions in big consulting firms. You will want to target each resume to the specific 
employer. For example, employers in tech will be interested in relevant projects and experience analyzing datasets. Be clear 
about how you developed those skills in courses or...

  [Doc 2] 984 chars | ~246 tokens
  Pre-requisites (One of these)
Scrimba is offering 20% off to roadmap users
on their AI Engineer course that covers this
roadmap in depth. Check them out!
AI Engineer
Introduction
What is an AI Engineer?
AI Engineer vs ML Engineer
LLMs
Inference
Training
Embeddings
Vector Databases
RAG
Prompt Enginee...

  [Doc 3] 247 chars | ~61 tokens
  you have any additional questions about AI use in the job search process, please reach out to our office. 
 
The resources this packet highlights

## 4.8 Debugging — Compression Metrics Across All Strategies

A production engineer must be able to measure the effectiveness of each compressor.  
Let's build a reusable debug function and run it across all three strategies.


In [30]:
def compression_debug_report(
    query: str,
    retriever,
    label: str,
) -> dict:
    """
    Run a retriever and return a structured compression metrics report.

    Returns a dict with: docs_in, docs_out, tokens_in, tokens_out, ratio, extracts.
    """
    # Baseline: what base_retriever fetches before compression
    original    = base_retriever.invoke(query)
    compressed  = retriever.invoke(query)

    tokens_in   = estimate_tokens(sum(len(d.page_content) for d in original))
    tokens_out  = estimate_tokens(sum(len(d.page_content) for d in compressed))
    ratio       = tokens_out / max(tokens_in, 1)

    print(f"{'─'*65}")
    print(f"  Strategy : {label}")
    print(f"  Query    : {query[:60]}...")
    print(f"  Docs in  : {len(original)}  →  Docs out: {len(compressed)}")
    print(f"  Tokens   : ~{tokens_in} → ~{tokens_out}  (ratio: {ratio:.2f})")
    print(f"  Saved    : ~{tokens_in - tokens_out} tokens ({(1-ratio)*100:.1f}% reduction)")
    if compressed:
        print(f"  Extract  : {compressed[0].page_content.strip()[:120]}...")
    else:
        print(f"  Extract  : [All documents filtered out]")
    print()

    return {
        "label": label, "docs_in": len(original), "docs_out": len(compressed),
        "tokens_in": tokens_in, "tokens_out": tokens_out, "ratio": ratio,
    }


debug_q = "What Python version and programming skills are required?"

print(f"\n=== Compression Debug Report ===\nQuery: {debug_q}\n")

report_base = compression_debug_report(debug_q, base_retriever,            "Base Retriever (no compression)")
report_ext  = compression_debug_report(debug_q, extractor_retriever,       "LLMChainExtractor")
report_flt  = compression_debug_report(debug_q, filter_retriever,          "LLMChainFilter")
report_emb  = compression_debug_report(debug_q, embedding_filter_retriever,"EmbeddingsFilter")



=== Compression Debug Report ===
Query: What Python version and programming skills are required?

─────────────────────────────────────────────────────────────────
  Strategy : Base Retriever (no compression)
  Query    : What Python version and programming skills are required?...
  Docs in  : 3  →  Docs out: 3
  Tokens   : ~692 → ~692  (ratio: 1.00)
  Saved    : ~0 tokens (0.0% reduction)
  Extract  : The UCC offers several options for getting help in creating and perfecting your resume.  Take advantage of 
one or more ...

─────────────────────────────────────────────────────────────────
  Strategy : LLMChainExtractor
  Query    : What Python version and programming skills are required?...
  Docs in  : 3  →  Docs out: 0
  Tokens   : ~692 → ~0  (ratio: 0.00)
  Saved    : ~692 tokens (100.0% reduction)
  Extract  : [All documents filtered out]

─────────────────────────────────────────────────────────────────
  Strategy : LLMChainFilter
  Query    : What Python version and programming

In [31]:
# ── Print comparison table ────────────────────────────────────────────────────
reports = [report_base, report_ext, report_flt, report_emb]

print(f"{'Strategy':<28} {'Docs In':>8} {'Docs Out':>9} {'Tok In':>7} {'Tok Out':>8} {'Ratio':>7}")
print("─" * 65)
for r in reports:
    print(
        f"{r['label']:<28} {r['docs_in']:>8} {r['docs_out']:>9} "
        f"{r['tokens_in']:>7} {r['tokens_out']:>8} {r['ratio']:>7.2f}"
    )
print()
print("Ratio < 1.0 = compression reduced context size.")
print("Ratio = 1.0 = no change (all docs kept).")


Strategy                      Docs In  Docs Out  Tok In  Tok Out   Ratio
─────────────────────────────────────────────────────────────────
Base Retriever (no compression)        3         3     692      692    1.00
LLMChainExtractor                   3         0     692        0    0.00
LLMChainFilter                      3         0     692        0    0.00
EmbeddingsFilter                    3         0     692        0    0.00

Ratio < 1.0 = compression reduced context size.
Ratio = 1.0 = no change (all docs kept).


## 4.9 Comparison — All Four Retrievers

Now we have four retrieval strategies. Let's run them all on multiple questions and compare.


In [32]:
comparison_questions = [
    "What Python version is required for the ML Engineer role?",
    "What are the career advancement paths in data science?",
    "How do I transition from software engineering to AI?",
]

print("=" * 75)
print(f"{'Question':<40} {'Base':>5} {'MQ':>4} {'PDR':>4} {'CCR':>5} {'CCR tok':>8}")
print(f"{'':40} {'docs':>5} {'docs':>4} {'docs':>4} {'docs':>5} {'(est)':>8}")
print("─" * 75)

for q in comparison_questions:
    base_d  = base_retriever.invoke(q)
    mq_d    = multi_query_retrieve(q, base_retriever, n_queries=3, verbose=False)
    pdr_d   = parent_doc_retriever.invoke(q)
    ccr_d   = extractor_retriever.invoke(q)
    ccr_tok = estimate_tokens(sum(len(d.page_content) for d in ccr_d))

    q_short = q[:39] + "…" if len(q) > 40 else q
    print(f"{q_short:<40} {len(base_d):>5} {len(mq_d):>4} {len(pdr_d):>4} {len(ccr_d):>5} {ccr_tok:>8}")

print("=" * 75)
print("MQ=MultiQueryRetriever  PDR=ParentDocumentRetriever  CCR=ContextualCompression")


Question                                  Base   MQ  PDR   CCR  CCR tok
                                          docs docs docs  docs    (est)
───────────────────────────────────────────────────────────────────────────
What Python version is required for the…     3    7    4     0        0
What are the career advancement paths i…     3    8    4     0        0
How do I transition from software engin…     3    8    3     0        0
MQ=MultiQueryRetriever  PDR=ParentDocumentRetriever  CCR=ContextualCompression


### Full Retriever Comparison Table

| Dimension | Base | MultiQuery | ParentDoc | CCR (LLM Extract) | CCR (EmbFilter) |
|---|---|---|---|---|---|
| **Retrieval unit** | Small chunk | Small chunk × N | Child → Parent | Any → Extracted | Any → Filtered |
| **Context quality** | ❌ Fragment | ❌ Fragments | ✅ Full context | ✅ Precise extract | ✅ Relevant docs |
| **Recall** | Low | High | Medium-High | Depends on base | Depends on threshold |
| **Precision** | High | Slightly lower | High | Very high | High |
| **Extra LLM calls** | 0 | 1 (query gen) | 0 | 1 per doc | 0 |
| **Token usage** | Low | Medium | High (parents large) | Lowest | Low |
| **Latency** | Fastest | Slow | Fast | Slowest | Fast |
| **RAM usage** | Low | Low | Medium (docstore) | Low | Low |
| **Vocabulary mismatch** | ❌ | ✅ | ❌ | ❌ | ❌ |
| **Context fragmentation** | ❌ | ❌ | ✅ | ✅ (+ precise) | Partial |
| **Best for** | Simple queries | Ambiguous queries | Context-heavy answers | High-cost production | Fast filtering |


## 4.10 Production Notes

### ✅ Use ContextualCompressionRetriever When
- LLM API cost is a primary concern — compression directly reduces tokens billed.
- Documents (or parent chunks) are large and contain mixed topics.
- Answer quality degrades because the LLM focuses on irrelevant paragraphs.
- You're building a production system with a context window budget.

### ❌ Avoid When
- Your chunks are already small and focused (100–200 tokens) — compression adds cost with no gain.
- Latency is critical — `LLMChainExtractor` adds one extra LLM call **per retrieved document**.
- Your retriever is already precise — compressing already-relevant chunks can accidentally remove context.
- Budget is extremely tight — with `LLMChainExtractor`, a query that retrieves 5 docs costs 6 LLM calls total.

### Latency & Cost Reality

```
base_retriever (k=3):
  1 embedding call + 1 vector search
  → ~50ms

ContextualCompressionRetriever (LLMChainExtractor, k=3):
  1 embedding call + 1 vector search + 3 LLM extraction calls
  → ~50ms + 3 × [LLM latency]
  → Can be 5–15× slower than base_retriever

ContextualCompressionRetriever (EmbeddingsFilter, k=3):
  1 embedding call + 1 vector search + 3 cosine similarity checks
  → ~60ms (nearly as fast as base_retriever)
```

### Common Mistakes

```
❌ Using LLMChainExtractor with k=10
   → 10 sequential LLM calls per user query
   → Latency becomes unacceptable (10–30s per query)
   → Reduce k to 3–5 when using extraction

❌ Setting EmbeddingsFilter threshold too high (> 0.90)
   → Drops too many documents — recall collapses
   → Set between 0.70–0.85 for balanced filtering

❌ Applying compression to already-small chunks
   → The extractor may return "NO_OUTPUT" for every chunk
   → Entire context is dropped; LLM receives nothing

❌ Chaining multiple compression layers
   → Compress → compress → compress introduces cascading errors
   → One compression stage is almost always sufficient
```


## 4.11 Engineering Best Practices

**1. Choose the right compressor for your constraint:**
```
Latency-critical       → EmbeddingsFilter (no LLM call)
Quality-critical       → LLMChainExtractor (precise verbatim extraction)
Balanced               → LLMChainFilter (coarse-grain doc filtering)
```

**2. Pair with `ParentDocumentRetriever` for the best pipeline:**
```
Query
  │
  ▼
ParentDocumentRetriever   ← precise child search + full parent context
  │
  ▼
ContextualCompressionRetriever (EmbeddingsFilter or LLMChainExtractor)
  │
  ▼
LLM  ← receives only the relevant passage from the large parent chunk
```
This combination maximizes both context quality and token efficiency.

**3. Keep k small when using LLM-based extraction:**  
With `LLMChainExtractor`, every retrieved document costs one LLM call.  
Set `base_retriever = vector_db.as_retriever(search_kwargs={"k": 3})` — not 10.

**4. Monitor compression ratio in production:**  
A healthy compression ratio is 0.10–0.40 (10–40% of original tokens retained).  
If ratio consistently > 0.8, your base chunks are already too small — skip compression.  
If ratio consistently < 0.05, your similarity threshold is too aggressive — raise it.

**5. Test "NO_OUTPUT" edge cases:**  
`LLMChainExtractor` can return empty results if the LLM finds nothing relevant.  
Always handle `len(compressed_docs) == 0` gracefully — fall back to `base_retriever` if needed.


## 4.12 Summary

```
The Problem:
  Even good retrieval returns too much text.
  LLMs lose focus. Tokens cost money. Latency increases.

Contextual Compression (post-retrieval filtering):
  Retriever fetches candidate documents.
  Compressor reads each document + the query.
  Only relevant passages are passed to the LLM.
  Original wording is preserved — this is extraction, not summarization.

Three compressor strategies:
  LLMChainExtractor  → highest quality, extra LLM call per doc
  LLMChainFilter     → doc-level keep/discard, one LLM call per doc
  EmbeddingsFilter   → fastest, no LLM, cosine similarity threshold
```

---

**Retriever Decision Guide (updated):**

| Scenario | Strategy |
|---|---|
| Simple, precise queries | `base_retriever` |
| Ambiguous / multi-intent queries | `MultiQueryRetriever` |
| Context-fragmented answers | `ParentDocumentRetriever` |
| Both ambiguity + fragmentation | `MultiQueryRetriever(base=ParentDocRetriever)` |
| Too many tokens reaching the LLM | `ContextualCompressionRetriever` |
| Token cost + context quality | `ParentDocRetriever` + `ContextualCompression` |

---

**Next**: Part 5 — Ensemble Retriever & Hybrid Search  
*(Combining multiple retrievers using weighted score fusion for maximum coverage.)*


---

# Part 5 — SelfQueryRetriever


## 5.1 Why Similarity Search is Not Enough

So far, we have used **Semantic Similarity** (Vector Search).  
Semantic similarity is incredibly powerful at understanding *concepts* and *meaning*, but it fails completely at understanding **structured constraints**.

### The Problem

Imagine asking a database of movie reviews:  
> *"Find me great sci-fi movies released after 2020 with a rating over 8."*

If we rely on standard vector search:
1. It converts this query into an embedding.
2. It looks for documents with similar embeddings.
3. It might retrieve a document saying: *"This sci-fi movie was terrible, rating 3/10, released in 1990"* — simply because the words "sci-fi", "rating", and "released" are semantically close to the query!

**Vector Search cannot understand:**
- `year > 2020`
- `rating > 8`
- `author == 'John Doe'`
- `category == 'Finance'`

To solve this, we must combine unstructured semantic search with structured database filtering.


## 5.2 Understanding Metadata

Every document we ingest has two parts:

1. **Page Content (Unstructured)**: The actual text (e.g., "The quick brown fox..."). This is what gets embedded and searched via vector similarity.
2. **Metadata (Structured)**: A dictionary of key-value pairs attached to the document (e.g., `{'author': 'Alice', 'page': 4, 'category': 'HR'}`).

> **Crucial Rule:** Metadata is NOT embedded. It is stored alongside the embedding in the vector database and used exclusively for exact filtering (like a SQL `WHERE` clause).

Let's inspect the metadata that already exists in our vector store.


In [33]:
# ── Inspect existing metadata in our Vector Store ─────────────────────────────
raw_sample = vector_db.get(limit=10, include=['documents', 'metadatas'])

all_keys = set()
for meta in raw_sample['metadatas']:
    if meta:
        all_keys.update(meta.keys())

print(f"Metadata fields available in our Vector Store:\n{sorted(list(all_keys))}\n")

print("=== Example Document ===")
print(f"Page Content : {raw_sample['documents'][0][:100]}...")
print(f"Metadata     : {raw_sample['metadatas'][0]}")


Metadata fields available in our Vector Store:
['author', 'category', 'creationdate', 'creator', 'filename', 'moddate', 'page', 'page_label', 'producer', 'source', 'title', 'total_pages']

=== Example Document ===
Page Content : Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or...
Metadata     : {'total_pages': 3, 'category': 'interview', 'author': 'Allison Viverette', 'source': 'd:\\career-ai-agent\\data\\interview\\CMU_Behavioral_Interview_Guide.pdf', 'title': 'Consultant Handout Templates (Under-Grad-Combo) v2', 'moddate': '2024-05-22T15:58:46-04:00', 'creationdate': '2024-05-22T15:58:46-04:00', 'creator': 'Microsoft® Word for Microsoft 365', 'filename': 'CMU_Behavioral_Interview_Guide.pdf', 'producer': 'Microsoft® Word for Microsoft 365', 'page': 0, 'page_label': '1'}


## 5.3 Manual Metadata Filtering

Before introducing advanced tools, let's look at how we manually filter metadata using Chroma's standard API.

In Chroma, we can pass a `filter` dictionary directly to the similarity search.


In [34]:
# ── Manual Metadata Filtering Demo ───────────────────────────────────────────
query = "What are the common behavioral interview questions?"

# We manually specify a hardcoded filter: category must be 'interview'
manual_filter = {"category": "interview"}

print(f"Query  : {query}")
print(f"Filter : {manual_filter}\n")

# Run search WITH filter
filtered_docs = vector_db.similarity_search(query, k=3, filter=manual_filter)

for i, doc in enumerate(filtered_docs, 1):
    print(f"[Doc {i}] Category: {doc.metadata.get('category')} | Source: {doc.metadata.get('filename')}")
    print(f"  Content: {doc.page_content[:100].strip()}...")
    print()


Query  : What are the common behavioral interview questions?
Filter : {'category': 'interview'}

[Doc 1] Category: interview | Source: CMU_Behavioral_Interview_Guide.pdf
  Content: (Prepare responses to classic interview questions cont.) 
o General Interview Questions: Prior to fo...

[Doc 2] Category: interview | Source: CMU_Behavioral_Interview_Guide.pdf
  Content: with. These questions should be authentic to your curiosities and should not be 
easily answered by...

[Doc 3] Category: interview | Source: CMU_Behavioral_Interview_Guide.pdf
  Content: Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or...



**The Limitation:**  
Hardcoding filters in code is easy, but users don't type Python dictionaries. Users type natural language:  
*"Show me interview questions from the CMU guide."*

We need a system that can automatically translate natural language into structured metadata filters.


## 5.4 What is SelfQueryRetriever?

`SelfQueryRetriever` uses an LLM to parse a natural language query into two distinct parts:
1. **Semantic Query**: The part of the sentence that should be used for vector search.
2. **Metadata Filters**: The structured constraints that should be applied to the database.

It "queries itself" by converting human language into a structured database query.


## 5.5 Internal Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                       SelfQueryRetriever                            │
│                                                                     │
│  User Input: "Show me interview tips from the CMU guide"            │
│         │                                                           │
│         ▼                                                           │
│  ┌─────────────────┐                                                │
│  │   LLM Parser    │ ← Uses Metadata Schema to understand fields    │
│  └─────────────────┘                                                │
│         │                                                           │
│         ├─► Semantic Query: "interview tips"                        │
│         │                                                           │
│         └─► Metadata Filter: { "filename": { "$eq": "CMU_...pdf" }} │
│         │                                                           │
│         ▼                                                           │
│  ┌─────────────────┐                                                │
│  │  Vector Store   │ ← Applies filter FIRST, then vector search     │
│  └─────────────────┘                                                │
│         │                                                           │
│         ▼                                                           │
│  Filtered & Semantically Matched Documents                          │
└─────────────────────────────────────────────────────────────────────┘
```

| Component | Role |
|---|---|
| **LLM Parser** | Translates natural language into a structured query object. |
| **Metadata Schema** | Tells the LLM what fields exist in the DB so it knows what to extract. |
| **Vector Store** | Executes the combined query (Search + Filter). |


## 5.6 Building the Metadata Schema

For the LLM to write correct filters, we must define exactly what metadata fields exist in our vector database. We do this using `AttributeInfo`.

If the LLM doesn't know a field exists, it cannot filter by it.


In [35]:
from langchain_classic.chains.query_constructor.base import AttributeInfo

# ── Define the schema for our existing metadata ──────────────────────────────
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The category of the document. Examples: 'interview', 'resume', 'cover_letter', 'networking'",
        type="string",
    ),
    AttributeInfo(
        name="filename",
        description="The original name of the PDF file.",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The page number within the document (0-indexed).",
        type="integer",
    ),
    AttributeInfo(
        name="author",
        description="The author or creator of the document.",
        type="string",
    )
]

document_content_description = "Guides and resources for career preparation, interviewing, and resumes."

print("✅ Metadata schema defined.")
for info in metadata_field_info:
    print(f"- {info.name} ({info.type}): {info.description}")


✅ Metadata schema defined.
- category (string): The category of the document. Examples: 'interview', 'resume', 'cover_letter', 'networking'
- filename (string): The original name of the PDF file.
- page (integer): The page number within the document (0-indexed).
- author (string): The author or creator of the document.


## 5.7 Implement SelfQueryRetriever

Now we instantiate the retriever using our existing `llm`, `vector_db`, and the schema we just built.


In [36]:
import lark
print(lark.__file__)

from lark import Lark
print("OK")

d:\career-ai-agent\.venv\Lib\site-packages\lark\__init__.py
OK


In [37]:
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator

# ── Build the SelfQueryRetriever ─────────────────────────────────────────────
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_db,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(),
    verbose=True, # Enables logging of the generated query
)

print("✅ SelfQueryRetriever built successfully.")


✅ SelfQueryRetriever built successfully.


In [38]:
!pip install lark


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 5.8 Debugging — Observing the Translation

Let's test it with a query that mixes semantic search and structured constraints. We will enable logging to see exactly what the LLM generates.


In [39]:
import logging
logging.basicConfig(level=logging.INFO)

# ── Test Query with Semantic + Structured requirements ───────────────────────
complex_query = "What are the common behavioral questions? Only show me results from the interview category."

print(f"User Query: {complex_query}\n")
print(f"{'═'*65}")

# The retriever will print the parsed structured query to the console
sq_docs = self_query_retriever.invoke(complex_query)

print(f"{'═'*65}\n")
print(f"Documents Retrieved: {len(sq_docs)}\n")

for i, doc in enumerate(sq_docs, 1):
    print(f"[Doc {i}]")
    print(f"  Category : {doc.metadata.get('category')}")
    print(f"  Filename : {doc.metadata.get('filename')}")
    print(f"  Content  : {doc.page_content.strip()[:150]}...")
    print()


User Query: What are the common behavioral questions? Only show me results from the interview category.

═════════════════════════════════════════════════════════════════


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.self_query.base:Generated Query: query='common behavioral questions' filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='category', value='interview') limit=None


═════════════════════════════════════════════════════════════════

Documents Retrieved: 4

[Doc 1]
  Category : interview
  Filename : CMU_Behavioral_Interview_Guide.pdf
  Content  : mentioned previously, I am currently getting my master’s in Computer Vision at Carnegie Mellon, with an 
interest in spherical CNNs. I am very interes...

[Doc 2]
  Category : interview
  Filename : CMU_Behavioral_Interview_Guide.pdf
  Content  : (Prepare responses to classic interview questions cont.) 
o General Interview Questions: Prior to formal behavioral interview questions, an interview ...

[Doc 3]
  Category : interview
  Filename : CMU_Behavioral_Interview_Guide.pdf
  Content  : Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or more behavioral interviews 
with a company of int...

[Doc 4]
  Category : interview
  Filename : Meta_ML_onsite_interview_prep.pdf
  Content  : 6
• Embracing Ambiguity:  How do you operate in an ambiguous and quickly
chan

**What just happened?**
1. The LLM saw: *"What are the common behavioral questions? Only show me results from the interview category."*
2. It extracted the semantic intent: `"common behavioral questions"`
3. It extracted the metadata constraint: `category == "interview"`
4. It passed both to Chroma. Chroma filtered the DB to *only* include the interview category, then ran the semantic search.


## 5.9 Comparison — The Retrieval Ecosystem

We now have five distinct retrieval strategies. Here is how they compare:

| Retriever | Core Mechanism | Solves | Weakness |
|---|---|---|---|
| **Base Retriever** | Simple Vector Search | Baseline retrieval | Context loss, vocabulary mismatch |
| **MultiQuery** | LLM generates query variants | Vocabulary mismatch | Still suffers from context fragmentation |
| **ParentDocument** | Small embed → Large return | Context fragmentation | Returns too many irrelevant tokens |
| **Contextual Compression** | LLM extracts relevant sentences | High token cost & noise | Adds latency / LLM cost per doc |
| **SelfQuery** | LLM translates query to DB filters | Missing structured metadata | Fails if metadata schema is poor |

### When to use which?

- Use **ParentDocument** when answers span across paragraphs.
- Use **Contextual Compression** when cost is high and precision is critical.
- Use **SelfQuery** when users ask questions involving dates, authors, categories, or numbers (e.g., "Show me docs from 2023").


## 5.10 Production Notes & Best Practices

### ✅ Advantages
- Instantly adds structured filtering to natural language applications.
- Dramatically improves precision by eliminating entire categories of irrelevant documents before vector search even begins.
- Very fast execution (the LLM call is only to parse the query once, not per document).

### ❌ Limitations
- **Garbage In, Garbage Out**: If your ingestion pipeline didn't extract good metadata, SelfQuery is useless.
- **Hallucinated Filters**: Sometimes the LLM applies a filter that doesn't exist, returning zero results.
- **Schema Maintenance**: You must keep `AttributeInfo` perfectly synced with your database schema.

### Best Practices

```
1. Explicit Schema Descriptions
   Provide examples in your AttributeInfo descriptions.
   Instead of: "The category."
   Use: "The category of the doc. Valid options are: 'HR', 'Finance', 'Engineering'."

2. Use Enums if Possible
   If a field only has 4 valid options, state them explicitly in the prompt so the LLM doesn't guess.

3. Graceful Fallback
   If SelfQuery returns 0 documents, always fall back to a Base Retriever search.
   The LLM might have applied a filter that was too strict.
```


## 5.11 Preparing for the Next Chapter

SelfQueryRetriever is exceptional at enforcing structured constraints.

However, a fundamental truth in AI Engineering is that **no single retriever is perfect.**

- **Base Retriever** is great for exact semantic matches.
- **SelfQuery** is great when metadata is mentioned.
- **BM25 (Keyword Search)** is great for exact acronyms and product IDs that embeddings fail to understand.

Sometimes, one retriever misses critical information that another one finds effortlessly.

This naturally leads to our next architectural pattern: **Ensemble Retriever**.
Instead of choosing just one retriever, what if we run multiple retrievers in parallel and mathematically fuse their results using an algorithm like Reciprocal Rank Fusion (RRF)?

**Next**: Part 6 — Ensemble Retriever & Hybrid Search


---

# Part 6 — Ensemble Retriever


## 6.1 Why One Retriever Is Not Enough

We have explored five distinct retrievers. You might be wondering: *"Which one is the best?"*

The truth in AI Engineering is that **no single retriever is universally best.** Every retrieval strategy has a blind spot.

### The Weaknesses of Individual Retrievers

| Retriever | Strength | Weakness | Example Failure Case |
|---|---|---|---|
| **Vector Search (Base)** | Understands concepts & meaning | Terrible at exact keyword matching | Fails on "HTTP 404" (embeds it as generic "error") |
| **BM25 (Keyword Search)** | Perfect for exact terms/IDs | Cannot understand semantics/synonyms | Fails on "car" if the document says "automobile" |
| **ParentDocument** | Provides rich surrounding context | Retrieves too many irrelevant tokens | Dilutes LLM attention on simple factoid questions |
| **Contextual Compression** | Extremely high precision | Slow latency & high LLM cost | Too expensive to run on every user query |
| **SelfQuery** | Perfect for structured filters | Useless if metadata is missing | Fails if the query doesn't mention a filterable field |

If you rely solely on Vector Search, you will miss exact product codes.  
If you rely solely on BM25, you will miss semantic intent.

**The Solution:** Don't choose just one. Run multiple retrievers simultaneously and combine their results.


## 6.2 What is Ensemble Retrieval?

**Ensemble Retrieval** (often called Hybrid Search) is the process of executing multiple different retrievers in parallel, merging their retrieved documents, and ranking them to produce a final, superior list of documents.

```
       ┌──────────────┐     ┌──────────────┐
       │ Retriever A  │     │ Retriever B  │
       │ (e.g. BM25)  │     │ (e.g. Vector)│
       └──────┬───────┘     └───────┬──────┘
              │                     │
          [Docs A]               [Docs B]
              │                     │
              └─────────┬───────────┘
                        ▼
                Merge Documents
                        │
                        ▼
               Remove Duplicates
                        │
                        ▼
                Rank Documents
                        │
                        ▼
              Final Top-K Context
```


## 6.3 Manual Example

Let's build an intuition for how merging and ranking works before we use LangChain's built-in tools.

Imagine two retrievers looking for documents about "Python Async":
- **Retriever A (Vector)** returns: Doc 1, Doc 3, Doc 5
- **Retriever B (BM25)** returns: Doc 3, Doc 2, Doc 1

How do we decide the final top 3?


In [40]:
# ── Educational Example: Manual Ensemble Merging ─────────────────────────────

# Fake results from two retrievers (ordered by rank: 1st, 2nd, 3rd)
vector_results = ["Doc_1", "Doc_3", "Doc_5"]
bm25_results   = ["Doc_3", "Doc_2", "Doc_1"]

def manual_rrf(list_a: list[str], list_b: list[str], k: int = 60) -> list[str]:
    """
    Reciprocal Rank Fusion (RRF) algorithm.
    Formula: score = 1 / (rank + k)
    """
    scores = {}
    
    # Process List A
    for rank, doc in enumerate(list_a, start=1):
        scores[doc] = scores.get(doc, 0) + (1.0 / (rank + k))
        
    # Process List B
    for rank, doc in enumerate(list_b, start=1):
        scores[doc] = scores.get(doc, 0) + (1.0 / (rank + k))
        
    # Sort by descending score
    ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked_docs]

# Run manual fusion
final_ranking = manual_rrf(vector_results, bm25_results)

print("Vector Results :", vector_results)
print("BM25 Results   :", bm25_results)
print("="*40)
print("Final Ranked List :", final_ranking)
print("\nNotice how 'Doc_3' (which was 2nd in vector and 1st in BM25) is ranked highly!")


Vector Results : ['Doc_1', 'Doc_3', 'Doc_5']
BM25 Results   : ['Doc_3', 'Doc_2', 'Doc_1']
Final Ranked List : ['Doc_3', 'Doc_1', 'Doc_2', 'Doc_5']

Notice how 'Doc_3' (which was 2nd in vector and 1st in BM25) is ranked highly!


## 6.4 Internal Architecture

```
┌────────────────────────────────────────────────────────────────────────┐
│                        EnsembleRetriever                               │
│                                                                        │
│  User Query                                                            │
│       │                                                                │
│       ├─────────────────────────┬─────────────────────────┐            │
│       ▼                         ▼                         ▼            │
│  ┌─────────┐               ┌─────────┐               ┌─────────┐       │
│  │ Base    │ (weight 0.4)  │ BM25    │ (weight 0.4)  │ SelfQ   │ (0.2) │
│  │ Vector  │               │ Keyword │               │ Filter  │       │
│  └─────────┘               └─────────┘               └─────────┘       │
│       │                         │                         │            │
│       ▼                         ▼                         ▼            │
│  [d1, d3, d5]              [d3, d2, d1]              [d5, d9, d1]      │
│       │                         │                         │            │
│       └─────────────────────────┼─────────────────────────┘            │
│                                 ▼                                      │
│                        Merge & Deduplicate                             │
│                        (Unique: d1, d2, d3, d5, d9)                    │
│                                 │                                      │
│                                 ▼                                      │
│                   Apply Reciprocal Rank Fusion (RRF)                   │
│                   (Multiply by retriever weights)                      │
│                                 │                                      │
│                                 ▼                                      │
│                          Return Top K                                  │
└────────────────────────────────────────────────────────────────────────┘
```


## 6.5 Ranking Strategies

When you merge lists, you must score the documents to rank them.

1. **Simple Merge (Round Robin)**: Take 1st from A, 1st from B, 2nd from A, 2nd from B.
   - *Problem*: Ignores if a document was found by *both* retrievers.
2. **Score Aggregation**: Add the raw scores from Retriever A and Retriever B.
   - *Problem*: BM25 scores (e.g., 14.5) and Cosine Similarity scores (e.g., 0.85) are on completely different scales. You cannot mathematically add them.
3. **Reciprocal Rank Fusion (RRF)**: The industry standard.
   - It ignores raw scores entirely!
   - It only looks at the **Rank** (1st, 2nd, 3rd) of the document in each list.
   - Formula: `Score = 1 / (Rank + K)`
   - Documents found in multiple lists get their RRF scores added together, pushing them to the top.


## 6.6 LangChain EnsembleRetriever

We will combine two of the most popular retrievers:
1. Our existing **Vector Retriever** (for semantic understanding).
2. A new **BM25 Retriever** (for exact keyword matching).

> **Note**: BM25 is an in-memory algorithm. We will initialize it using the exact same documents that are currently inside our Vector Database so they are searching the same corpus.


In [41]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# ── 1. Create the BM25 Retriever ─────────────────────────────────────────────
# We extract the documents already sitting in our vector database
raw_data = vector_db.get(include=["documents", "metadatas"])
docs_for_bm25 = [
    Document(page_content=doc, metadata=meta)
    for doc, meta in zip(raw_data["documents"], raw_data["metadatas"])
]

# Initialize BM25 with our existing documents
bm25_retriever = BM25Retriever.from_documents(docs_for_bm25)
# Set k=3 so it returns 3 documents per query
bm25_retriever.k = 3

print(f"✅ BM25Retriever initialized with {len(docs_for_bm25)} documents.")

# ── 2. Create the Ensemble Retriever ─────────────────────────────────────────
# We combine Vector (base_retriever) and Keyword (bm25_retriever)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, base_retriever],
    weights=[0.5, 0.5]  # 50% importance to BM25, 50% to Vector Search
)

print("✅ EnsembleRetriever built successfully.")


✅ BM25Retriever initialized with 285 documents.
✅ EnsembleRetriever built successfully.


## 6.7 Debugging — Tracing the Fusion

Let's run a query that benefits from both keyword matching and semantic understanding, and trace exactly how the Ensemble Retriever merges the lists.


In [42]:
query = "What is the STAR format for answering questions?"

print(f"Query: {query}\n")

# 1. Run BM25 independently
print("=== BM25 Results ===")
bm25_docs = bm25_retriever.invoke(query)
for i, d in enumerate(bm25_docs, 1):
    print(f"[{i}] {d.page_content[:60].strip()}... (Source: {d.metadata.get('filename')})")

print("\n=== Vector Results ===")
# 2. Run Vector independently
vector_docs = base_retriever.invoke(query)
for i, d in enumerate(vector_docs, 1):
    print(f"[{i}] {d.page_content[:60].strip()}... (Source: {d.metadata.get('filename')})")

print("\n=== ENSEMBLE RESULTS (Fused) ===")
# 3. Run Ensemble
ensemble_docs = ensemble_retriever.invoke(query)
for i, d in enumerate(ensemble_docs, 1):
    # Determine which retrievers found this doc
    found_in = []
    if any(d.page_content == bd.page_content for bd in bm25_docs): found_in.append("BM25")
    if any(d.page_content == vd.page_content for vd in vector_docs): found_in.append("Vector")
    
    print(f"[Rank {i}] Found by: {' + '.join(found_in)}")
    print(f"  Content: {d.page_content[:80].strip()}...")


Query: What is the STAR format for answering questions?

=== BM25 Results ===
[1] capable and qualified candidate. Smiling throughout the inte... (Source: CMU_Behavioral_Interview_Guide.pdf)
[2] (Prepare responses to classic interview questions cont.) 
o... (Source: CMU_Behavioral_Interview_Guide.pdf)
[3] git log
show the commit history for the currently active bra... (Source: git-cheat-sheet-education.pdf)

=== Vector Results ===
[1] • Use easy-to-read, common fonts.  
• Avoid graphic resumes... (Source: ResumeGuideforStudentsGeneral.pdf)
[2] allow the reader’s eye to rest. Using 0.7-inch to 1-inch mar... (Source: ResumeGuideforStudentsGeneral.pdf)
[3] what will make you stand out. The information you choose to... (Source: ResumeGuideforStudentsGeneral.pdf)

=== ENSEMBLE RESULTS (Fused) ===
[Rank 1] Found by: BM25
  Content: capable and qualified candidate. Smiling throughout the interview will not only...
[Rank 2] Found by: Vector
  Content: • Use easy-to-read, common fonts.  
• Avoi

**Observation**: 
Documents found by *both* retrievers are boosted to the top (Rank 1). Documents found by only one retriever fill out the rest of the top K. Duplicates are removed automatically.


## 6.8 Comparisons

Where does Ensemble fit in the overall ecosystem?

| Metric | Base Vector | BM25 | MultiQuery | ParentDoc | Compression | Ensemble (Vector+BM25) |
|---|---|---|---|---|---|---|
| **Semantic Search** | ✅ Yes | ❌ No | ✅ Yes | ✅ Yes | ✅ Yes | ✅ Yes |
| **Keyword Search** | ❌ No | ✅ Yes | ❌ No | ❌ No | ❌ No | ✅ Yes |
| **Metadata Filtering**| ❌ No | ❌ No | ❌ No | ❌ No | ❌ No | ❌ No (Unless Base=SelfQuery) |
| **Recall** | Low | Low | High | Medium | Low | **Highest** |
| **Latency** | Fast | Fast | Slow | Fast | Slowest | Medium (Parallel execution) |
| **LLM Calls** | 0 | 0 | 1 | 0 | 1 per doc | 0 |

**Best Use Case for Ensemble**: Any production RAG system. Relying purely on Vector Search is considered a beginner mistake in modern AI Engineering. You almost always want a BM25 + Vector hybrid.


## 6.9 Production Notes & Best Practices

### ✅ Advantages
- **Maximum Coverage**: Covers both semantic queries ("how do I fix my screen") and exact keyword queries ("Error Code 0x800F081F").
- **Zero LLM Cost**: Fusing algorithms like RRF run locally in Python. No LLM tokens required.

### ❌ Disadvantages
- **Memory/Infrastructure**: BM25 requires keeping a sparse index in memory, or running a dedicated search engine like Elasticsearch/Pinecone.
- **Context Window Pressure**: If BM25 returns 3 docs and Vector returns 3 docs, and there is no overlap, you send 6 documents to the LLM. You must carefully manage your `k` parameters.

### Choosing Weights
- Start with `weights=[0.5, 0.5]`.
- If your users search for lots of names, product codes, or acronyms, increase BM25: `weights=[0.7, 0.3]`.
- If your users ask long, conversational questions, increase Vector: `weights=[0.3, 0.7]`.

### When NOT to use Ensemble
- When latency is absolutely critical and running BM25 alongside Vector exceeds your time budget (rare).
- When your corpus is incredibly small (e.g., 5 pages). Vector search alone is sufficient.


## 6.10 Mini Exercises

Try modifying the code block above to implement these architectures:

1. **The "Precision Heavy" Ensemble**: 
   Combine `bm25_retriever` with `extractor_retriever` (Contextual Compression).
   *Expected outcome*: BM25 finds exact docs, Compression extracts only the specific sentences. Extremely high quality, but high latency.

2. **The "Structured Hybrid" Ensemble**:
   Combine `bm25_retriever` with `self_query_retriever`.
   *Expected outcome*: Can handle exact keyword matching AND complex metadata filters simultaneously.

3. **Weight Tuning**:
   Set `weights=[0.9, 0.1]` (Heavy BM25 bias). Query for a concept without using the exact words in the text. Observe how the Vector results get pushed down the ranking.



## 6.11 Preparing for Part 7

We have built an incredible retrieval pipeline. By combining BM25 and Vector Search using the `EnsembleRetriever`, we are retrieving highly relevant documents almost every time.

**But we have hit a ceiling.**

Reciprocal Rank Fusion (RRF) is just a math formula. It looks at the *rank* of the documents, but it **does not actually read the text**.

If BM25 ranks a terrible document as #1, and Vector Search ranks a terrible document as #1, RRF will happily boost those terrible documents to the top of the final list.

Instead of mathematically guessing which document is best based on list positions... what if we used a specialized, lightweight AI model to **read** all the retrieved documents and **score** them based on how well they actually answer the user's question?

This brings us to the final piece of the modern RAG architecture: **Rerankers**.

**Next**: Part 7 — Rerankers


---

# Part 7 — Rerankers


## 7.1 Why Ensemble Retriever is Still Not Enough

By combining Vector Search and BM25 into an `EnsembleRetriever` (Part 6), we solved the **Recall** problem. We are now almost guaranteed to retrieve the correct document in the Top-K results because we search by both meaning and exact keywords.

However, we have introduced a new problem: **Precision and Ordering**.

Reciprocal Rank Fusion (RRF) merges the results based purely on their *math rank*. It does not actually read the text! If BM25 returns a terrible document at rank #1, and Vector Search returns a terrible document at rank #1, RRF will confidently boost those terrible documents to the very top.

### Retrieval vs Ranking (The Two-Stage Pipeline)
1. **Candidate Generation (Retrieval)**: Fast, cheap, and sloppy. The goal is High Recall (fetch 50 documents, hoping the answer is in there). Vector Search and BM25 live here.
2. **Scoring (Ranking)**: Slow, expensive, and accurate. The goal is High Precision. We take the 50 documents from step 1, read them carefully, and reorder them so the best 5 go to the LLM. Rerankers live here.


## 7.2 Why Vector Search Misorders Documents

Why do we need a Reranker if the Vector Database already scores documents by cosine similarity?

Vector databases use **Bi-Encoders**.
A Bi-Encoder embeds the Query into a vector, embeds the Document into a vector, and compares the angle between them. 
- *Limitation*: The document was embedded *months ago* during ingestion, completely unaware of what the future user query would be.
- *Failure Case*: 
  - Query: *"How do I cancel my subscription?"*
  - Doc A: *"To upgrade your subscription, click here."* (Highly semantically similar! High cosine score).
  - Doc B: *"Steps to terminate your account and stop billing."* (Different vocabulary, lower cosine score, but it is the actual answer!).

Nearest neighbors in a vector space indicate *topic similarity*, not *answer relevance*.


## 7.3 The Solution: Cross Encoders

A Reranker uses a **Cross-Encoder** architecture.

Unlike a Bi-Encoder which embeds the query and document separately, a Cross-Encoder concatenates the Query and the Document together and passes them simultaneously through the Transformer layers.

```
BI-ENCODER (Fast, used in Vector DBs)
Query ────► Embedding ─┐
                       ├──► Cosine Similarity
Doc   ────► Embedding ─┘

CROSS-ENCODER (Slow, used for Reranking)
[Query + Doc] ────► Transformer Layers ────► Relevance Score (0.0 to 1.0)
```

**Why is it more accurate?**
Because the Transformer's attention mechanism can look at the words in the Document *in the direct context* of the words in the Query. It can learn that "terminate" in the document explicitly answers "cancel" in the query.

**Why don't we use it for everything?**
Computational complexity. If you have 1 million documents, running a Cross-Encoder for every user query would take hours. We must use Bi-Encoders to quickly filter down to the top 50, and then use Cross-Encoders to rank those 50.


## 7.4 The Modern Two-Stage Retrieval Pipeline

```
     User Query
         │
         ▼
 ┌───────────────┐
 │  Retriever    │  ◄── Vector Search / BM25 / Ensemble (Fast)
 └───────────────┘
         │
         ▼
 [Top 50 Documents] ◄── High Recall, Low Precision, Misordered
         │
         ▼
 ┌───────────────┐
 │   Reranker    │  ◄── Cross-Encoder (Reads Query + Doc)
 └───────────────┘
         │
         ▼
 [Top 5 Documents]  ◄── High Precision, Perfectly Ordered
         │
         ▼
 ┌───────────────┐
 │     LLM       │  ◄── Generates Final Answer
 └───────────────┘
```


## 7.5 Popular Rerankers

| Reranker | Deployment | Speed | Cost | Notes |
|---|---|---|---|---|
| **Cohere Rerank 3** | API | Fast | Paid ($) | The industry standard API. Multilingual, extremely accurate. |
| **Jina AI Reranker** | API / Local | Fast | Paid / Free | Excellent open-source models (v2), very fast. |
| **BAAI/bge-reranker-v2** | Local (HuggingFace) | Varies by GPU | Free | The most popular open-source family. Excellent accuracy. |
| **Mixedbread** | Local (HuggingFace) | Very Fast | Free | Highly optimized for latency. Great for CPU deployments. |

For this notebook, we will use an open-source HuggingFace model (`BAAI/bge-reranker-base`) so it runs entirely on your local machine.


## 7.6 LangChain Integration

In LangChain, a Reranker is implemented as a **Document Compressor**. It takes a list of documents, scores them, sorts them, and "compresses" the list by only returning the Top N.

Let's implement a custom `CrossEncoderReranker` to see exactly how the scoring works under the hood.


In [46]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.documents import BaseDocumentCompressor
from langchain_core.callbacks import Callbacks
from langchain_core.documents import Document
from typing import Sequence, Optional, Any
from pydantic import ConfigDict

# 1. Load the Cross-Encoder model (we use a lightweight model for speed)
cross_encoder_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
print("✅ Cross-Encoder model loaded.")

# 2. Build a Custom Reranker Compressor
class CustomCrossEncoderReranker(BaseDocumentCompressor):
    model: Any
    top_n: int = 3
    
    # Allow arbitrary types for the model field
    model_config = ConfigDict(arbitrary_types_allowed=True)

    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Optional[Callbacks] = None,
    ) -> Sequence[Document]:
        if not documents:
            return []
        
        # Prepare pairs of (Query, Document)
        text_pairs = [[query, doc.page_content] for doc in documents]
        
        # The Cross-Encoder scores the pairs
        scores = self.model.score(text_pairs)
        
        # Zip documents with their scores and sort descending
        scored_docs = list(zip(documents, scores))
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        
        # Update metadata with the score and return the Top N
        top_docs = []
        for doc, score in scored_docs[:self.top_n]:
            doc.metadata["relevance_score"] = float(score)
            top_docs.append(doc)
            
        return top_docs

# Instantiate the reranker
reranker_compressor = CustomCrossEncoderReranker(model=cross_encoder_model, top_n=3)
print("✅ Custom Reranker Compressor built.")


INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
INFO:sentence_transformers.base.model:No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter

✅ Cross-Encoder model loaded.
✅ Custom Reranker Compressor built.


ERROR: Could not find a version that satisfies the requirement langchain_retrievers (from versions: none)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for langchain_retrievers


### Wrapping the Retriever

Now we wrap our `base_retriever` (which returns 10+ documents) with the `ContextualCompressionRetriever`. It will fetch the 10 documents, pass them to the Reranker, and output the best 3.


In [47]:
from langchain_classic.retrievers import ContextualCompressionRetriever

# We will increase the base retriever to fetch 10 documents
# This gives the Reranker a larger pool of candidates to choose from
base_retriever.search_kwargs["k"] = 10

# Wrap the retriever
reranked_retriever = ContextualCompressionRetriever(
    base_compressor=reranker_compressor,
    base_retriever=base_retriever
)

print("✅ Two-Stage Retrieval Pipeline ready (Base Retriever -> Reranker).")


✅ Two-Stage Retrieval Pipeline ready (Base Retriever -> Reranker).


## 7.7 Debugging — Observing the Reordering

Let's run a query and print the documents *before* and *after* reranking to see how the Cross-Encoder fixes Vector Search mistakes.


In [48]:
query = "What is the best way to structure an answer in a behavioral interview?"

print(f"Query: {query}\n")

# 1. Fetch Candidates (Vector Search only)
print("=== Stage 1: Vector Search Candidates (Top 5 of 10) ===")
candidates = base_retriever.invoke(query)
for i, doc in enumerate(candidates[:5], 1):
    print(f"[{i}] {doc.page_content[:75].strip()}...")

print("\n" + "="*50 + "\n")

# 2. Rerank Candidates
print("=== Stage 2: Reranked Results (Top 3) ===")
reranked_docs = reranked_retriever.invoke(query)
for i, doc in enumerate(reranked_docs, 1):
    score = doc.metadata.get('relevance_score', 0)
    print(f"[Rank {i}] (Score: {score:.4f})")
    print(f"  Content: {doc.page_content[:100].strip()}...")
    
# Reset base retriever to normal after demo
base_retriever.search_kwargs["k"] = 3


Query: What is the best way to structure an answer in a behavioral interview?

=== Stage 1: Vector Search Candidates (Top 5 of 10) ===
[1] Your interviewer will be thinking about how your skills and experience migh...
[2] several different algorithms and understanding the tradeoffs is helpful. Fo...
[3] SKILLS  
Programming: Python (numpy, pandas, scikit-learn, pytorch), SQL, R...
[4] 4
Machine Learning (ML) System Design Interview
What can you expect?
The ML...
[5] Behavioral Interview Guide  
Overview: 
Throughout an interview process, yo...


=== Stage 2: Reranked Results (Top 3) ===


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]

[Rank 1] (Score: 0.3107)
  Content: (Prepare responses to classic interview questions cont.) 
o General Interview Questions: Prior to fo...
[Rank 2] (Score: 0.0705)
  Content: several different algorithms and understanding the tradeoffs is helpful. For
example, be able to exp...
[Rank 3] (Score: 0.0673)
  Content: Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or...


**Observation:**
Notice how a document that Vector Search placed at Rank #3 or #4 (because the raw wording was slightly different) gets boosted to Rank #1 by the Reranker because the Cross-Encoder actually *understands* that it answers the question.


## 7.8 Production Discussion & Best Practices

### Choosing K (Retrieval) and N (Reranking)
- **Top-K (Candidates)**: Usually set between `20` and `100`. Fetching 100 documents from Chroma takes milliseconds.
- **Top-N (Final)**: Usually set between `3` and `7`. Sending more than 7 chunks to an LLM often causes the "Lost in the Middle" phenomenon where the LLM forgets the context.

### Latency Tradeoffs
Cross-Encoders are neural networks. If you set `K=1000`, your system will take seconds (or minutes on CPU) to score 1000 pairs. 
- **CPU Deployments**: Keep `K <= 20`. Use lightweight models (`bge-reranker-base` or `bge-reranker-v2-m3`).
- **GPU Deployments**: You can safely push `K = 100`.

### The "Lost in the Middle" Problem
LLMs are great at reading the very beginning and very end of their prompt, but they ignore the middle. Rerankers solve this by putting the highest scoring, absolute most relevant chunk at the very top of the context window.


## 7.9 Comparisons — The Complete Retrieval Ecosystem

We have reached the pinnacle of retrieval strategies. Here is the final comparison of everything we have built in this notebook:

| Strategy | Recall | Precision | Latency | Infrastructure Cost | Primary Use Case |
|---|---|---|---|---|---|
| **Base Vector** | Low | Low | Very Fast | Low | Simple hobby projects |
| **MultiQuery** | High | Low | Slow (LLM) | High (Tokens) | Ambiguous / brief user queries |
| **ParentDoc** | Med | Med | Fast | Low | Documents where answers span paragraphs |
| **SelfQuery** | Med | High | Slow (LLM) | High (Tokens) | E-commerce, heavily structured DBs |
| **Ensemble (BM25+Vec)** | **Highest** | Low | Fast | Med (RAM for BM25) | Broad coverage. The best candidate generator. |
| **Reranker** | N/A | **Highest** | Med (GPU/API) | Med (Compute) | **Mandatory** final step in Production RAG |


## 7.10 Best Production Pipelines

If you are building an AI Agent for production tomorrow, which architecture should you use?

### 1. The Standard Production Architecture (90% of Use Cases)
```text
BM25 + Dense Retriever ──► Ensemble ──► Reranker ──► LLM
```
*Why?* Ensemble guarantees you find the document (High Recall). Reranker guarantees it's sent to the LLM in the correct order (High Precision).

### 2. The Heavy-Duty Architecture (For Complex Documents like Legal/Medical)
```text
ParentDocumentRetriever ──► Ensemble (with BM25) ──► Reranker ──► LLM
```
*Why?* Medical terminology requires BM25. Legal clauses span across pages (ParentDoc). The Reranker ensures the LLM isn't flooded with irrelevant laws.

### 3. The E-Commerce / Database Architecture
```text
SelfQueryRetriever ──► Reranker ──► LLM
```
*Why?* Users ask "Show me laptops under $500". Vector search fails at numbers. SelfQuery filters the DB down to $500 laptops, then the Reranker sorts by semantic relevance to "laptop for gaming".


## 7.11 Preparing for the Final Architecture

We have spent this entire notebook mastering **Retrieval**.
We can now fetch documents, filter by metadata, handle vocabulary mismatches, combine multiple search engines, and perfectly rerank the results.

But what happens when the user asks a question that requires multiple steps? 
What if they ask: *"Compare the interview strategies in the CMU guide with the resume strategies in the Stanford guide."*
A single retriever cannot answer this. We need an autonomous system that can call the retriever multiple times, read the results, decide if it needs more information, and synthesize an answer.

We must move beyond static chains. We must build **Agents**.

**Next Part: Production RAG Architecture**


---

# Part 8 — Production RAG

## 8.1 Introduction to Production RAG

We have explored multiple retrieval strategies. But how do we combine them into a resilient, fast, and cost-effective production system?

**Production RAG** is entirely different from local prototype RAG. In production, you will face:
- Mixed user intent (some ask for facts, some ask to summarize).
- Latency requirements (responses must start streaming in <1s).
- API failures and rate limits.
- High costs for embedding and LLM inference.

## 8.2 Query Routing

**Motivation:** Not every user query needs a Reranker. Not every user query needs the LLM!
- "What is your refund policy?" → Needs RAG.
- "Hi" → Does NOT need RAG.
- "Compare product A and B" → Needs MultiQuery RAG.

**Intuition:** A Query Router sits at the very beginning of the pipeline. It reads the user query and *routes* it to the correct retrieval strategy (or skips retrieval entirely) to save time and money.

### Architecture
```text
                  User Query
                      │
               ┌──────┴──────┐
               │ Query Router│
               └──────┬──────┘
       ┌──────────────┼──────────────┐
       ▼              ▼              ▼
  No Retrieval    Ensemble RAG   SelfQuery RAG
  (Chit-chat)   (Knowledge Base) (Database/E-commerce)
```


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

# ── Query Router Example ─────────────────────────────────────────────────────

class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: str = Field(
        ...,
        description="Given a user query choose to route it to 'web_search', 'vectorstore', or 'greeting'."
    )

router_llm = llm.with_structured_output(RouteQuery)

system = """You are an expert at routing a user query to the appropriate datasource.
Based on the programming language or concept, route it to the relevant datasource."""
route_prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{question}")]
)
question_router = route_prompt | router_llm

print("Router Test (Greeting):", question_router.invoke({"question": "hello there"}))
print("Router Test (Knowledge):", question_router.invoke({"question": "how do I format my resume"}))


## 8.3 Caching & Cost Optimization

**Motivation:** Users frequently ask the same questions (e.g., "What are the common behavioral interview questions?"). Running the LLM and Embeddings for the exact same query is a waste of money.

**Intuition:** We can cache the LLM response or cache the Vector Search results. If the query is identical, we instantly return the cached answer.

In LangChain, this is done natively using `set_llm_cache`.


In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
import time

# ── LLM Caching Example ──────────────────────────────────────────────────────
set_llm_cache(InMemoryCache())

print("Testing Cache...")

# First call (Slow - hits the API)
start = time.time()
llm.invoke("What is a resume? Give a 1 sentence summary.")
end1 = time.time() - start
print(f"First Call : {end1:.2f} seconds")

# Second call (Fast - hits the cache)
start = time.time()
llm.invoke("What is a resume? Give a 1 sentence summary.")
end2 = time.time() - start
print(f"Second Call: {end2:.2f} seconds (Cached!)")

# Clear the cache to prevent interference with other cells
set_llm_cache(None)


## 8.4 Retrieval Evaluation

**Motivation:** How do you know if adding a Reranker actually improved your system? You cannot guess. You must measure.

**Intuition:** We use evaluation frameworks (like RAGAS or LangSmith) to score retrieved documents on **Context Precision** and **Context Recall**.

*We will explore Evaluation deeply in the LangSmith section.*

## 8.5 Production Best Practices

### ✅ 1. Decouple Chunking from Retrieval (ParentDocument)
Never embed 1000-token chunks. Embed 200-token chunks and return the parent document.

### ✅ 2. Always Hybrid Search (Ensemble)
Never rely on Vector Search alone. BM25 is free and catches keyword mismatches.

### ✅ 3. Always Compress (Reranker)
Never dump 10 raw documents into the LLM context window. Use a Cross-Encoder to order them, and only pass the top 3.

### ❌ Common Mistakes
- **Blindly scaling K**: Increasing `K` from 5 to 50 will drastically increase latency and cost, and will confuse the LLM (Lost in the Middle).
- **Ignoring Caching**: 20% of user queries in production are often duplicates.


---

# Part 9 — RunnableWithMessageHistory

## 9.1 The Need for Memory

**Motivation:** Up until now, every time we invoked our RAG chain, the LLM had total amnesia. If you ask "What is a resume?" and then ask "How do I format *it*?", the LLM will fail because it does not know what "it" refers to.

**Intuition:** In a production application, multiple users are chatting simultaneously. The backend must remember the chat history for each specific user session and pass it to the LLM during every turn.

## 9.2 RunnableWithMessageHistory

LangChain provides `RunnableWithMessageHistory` to automatically manage and inject chat histories based on a `session_id`.

### Architecture
```text
 User Request (session="user_123")
        │
        ▼
 ┌─────────────┐     Fetches history for "user_123"
 │ Message     │ ◄── from Redis / Postgres / SQLite
 │ History     │
 │ Wrapper     │ ──► Appends to Prompt ──► LLM ──► Output
 └─────────────┘
        │
        ▼
 Saves new Human+AI messages to database
```


In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ── 1. Create an in-memory dictionary to act as our Database ───────────────
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# ── 2. Create the Chain ──────────────────────────────────────────────────────
# Notice we use MessagesPlaceholder to inject the history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

chain = prompt | llm

# ── 3. Wrap the Chain ────────────────────────────────────────────────────────
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)
print("✅ RunnableWithMessageHistory created.")


## 9.3 Simulating Multiple Users

Let's simulate two different users chatting with our application at the same time.


In [ ]:
# User A starts a conversation
print("=== User A (Session 1) ===")
res_a1 = with_message_history.invoke(
    {"question": "Hi, I'm Alice. I'm struggling with behavioral interviews."},
    config={"configurable": {"session_id": "session_A"}}
)
print("AI:", res_a1.content[:100], "...")

# User B starts a different conversation
print("\n=== User B (Session 2) ===")
res_b1 = with_message_history.invoke(
    {"question": "Hello, my name is Bob. I need a resume template."},
    config={"configurable": {"session_id": "session_B"}}
)
print("AI:", res_b1.content[:100], "...")

# User A asks a follow-up (The AI must remember Alice)
print("\n=== User A (Follow-up) ===")
res_a2 = with_message_history.invoke(
    {"question": "What was my name again, and what did I need help with?"},
    config={"configurable": {"session_id": "session_A"}}
)
print("AI:", res_a2.content)


## 9.4 Production Notes & Best Practices

### ✅ Advantages
- Instantly adds multi-turn conversation support to any chain.
- Handles multiple concurrent users automatically via `session_id`.

### ❌ Disadvantages
- **Context Window Overflow**: If a conversation goes on for 100 turns, you will exceed the LLM's maximum token limit and pay massive fees.

### Best Practices
- **Never use `ChatMessageHistory` in production**. It is strictly in-memory. If your server restarts, all users lose their chat history.
- **Use Redis or Postgres**: Use `RedisChatMessageHistory` or `PostgresChatMessageHistory` so state is persisted across server reboots.
- **Truncate History**: Always implement logic to keep only the last `N` messages (e.g., last 10 messages) or summarize older messages to save tokens.


---

# Part 10 — Advanced Execution: Streaming & Async

In a production environment, UX and backend scalability are critical. Users will not stare at a blank screen for 10 seconds waiting for an LLM. Your server cannot block execution while waiting for an external API.

## 10.1 Streaming

**Motivation:** An LLM might take 5 seconds to generate a full 500-word response. If you wait until it is finished to show the user, the application feels broken.

**Intuition:** We can stream the response token-by-token back to the user as it is being generated, reducing perceived latency to ~200ms.

LangChain provides three streaming methods:
1. `.stream()`: Streams the final output.
2. `.astream()`: The asynchronous version of `.stream()`.
3. `.astream_events()`: Streams *everything* happening in the chain (e.g., when the retriever starts, when the LLM starts, when tools are called). Highly advanced.


In [ ]:
import sys
import time

# ── Streaming Demo ───────────────────────────────────────────────────────────
print("User: Tell me a 3 sentence story about a programmer.\n")
print("AI: ", end="")

start = time.time()
first_token_time = None

# Using .stream() to get tokens as they are generated
for chunk in llm.stream("Tell me a 3 sentence story about a programmer."):
    if not first_token_time:
        first_token_time = time.time() - start
    # Print the chunk text immediately without a newline
    print(chunk.content, end="")
    sys.stdout.flush()

total_time = time.time() - start
print(f"\n\n[Metrics] Time to first token: {first_token_time:.2f}s | Total time: {total_time:.2f}s")


## 10.2 Async Execution

**Motivation:** Python is synchronous by default. If 10 users query your RAG application at the same time, a synchronous server will process User 1, wait for OpenAI to reply, process User 2, wait, etc. User 10 will wait a very long time.

**Intuition:** By using `asyncio` and LangChain's async methods (`ainvoke`, `astream`, `abatch`), the Python server can send 10 requests to the LLM simultaneously and handle other work while waiting for the responses.

### Synchronous vs Asynchronous Batching
Let's see the performance difference when translating 3 sentences.


In [ ]:
import asyncio
import time

queries = [
    "Translate 'Hello' to French.",
    "Translate 'Goodbye' to Spanish.",
    "Translate 'Thank you' to German."
]

# ── 1. Synchronous Execution ─────────────────────────────────────────────────
start_sync = time.time()
sync_results = llm.batch(queries)
end_sync = time.time() - start_sync
print(f"Synchronous .batch() took : {end_sync:.2f} seconds")

# ── 2. Asynchronous Execution ────────────────────────────────────────────────
async def run_async_batch():
    start_async = time.time()
    async_results = await llm.abatch(queries)
    end_async = time.time() - start_async
    print(f"Asynchronous .abatch() took: {end_async:.2f} seconds")
    return async_results

# In Jupyter, we can await directly in the cell
await run_async_batch()
print("\n✅ Notice how asynchronous execution is drastically faster because it processes all requests concurrently.")


## 10.3 Production Notes & Best Practices

- **Always use `astream()` in production endpoints**: If you are using FastAPI, return a `StreamingResponse` using an async generator wrapping `chain.astream()`.
- **Concurrency Limits**: When using `abatch()`, beware of API rate limits. Sending 500 concurrent requests to an LLM provider will likely result in a `429 Too Many Requests` error.
- **`astream_events()`**: If you are building a complex Agent that uses tools, and you want the UI to show "Agent is searching the web..." before streaming the text, you *must* use `astream_events()` to catch the tool-start signals.


---

# Part 11 — LangSmith & Production Workflow

## 11.1 What is LangSmith?

**Motivation:** You have built an incredible RAG pipeline (BM25 + Vector Ensemble → Reranker → LLM). You deploy it. A user complains: *"The bot gave me the wrong answer!"*
How do you debug this? Was the error caused by the Vector Search? The Reranker dropping the right document? The LLM hallucinating?

**Intuition:** LangSmith is a unified DevOps platform for LLM applications. It provides:
1. **Tracing**: A visual tree of every single step in your chain, showing exactly what went in and out of the Retriever, the Reranker, and the LLM.
2. **Evaluations**: Datasets to test your RAG pipeline. If you tweak your chunk size, LangSmith will run your pipeline against 100 test questions and score the new accuracy.
3. **Monitoring**: Dashboards showing latency, token costs, and user feedback (thumbs up/down).

## 11.2 Enabling LangSmith Tracing

Tracing is enabled purely via Environment Variables. No code changes are required!

If you set these variables in your `.env` file, every `.invoke()` will be sent to the LangSmith dashboard.

```python
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_api_key"
os.environ["LANGCHAIN_PROJECT"] = "career-ai-agent-v1"
```

*Note: We explicitly disabled tracing at the top of this notebook to prevent clutter, but in a production environment, it is absolutely essential.*

## 11.3 The Production AI Engineering Workflow

AI Engineering is highly iterative. The standard workflow using LangSmith is:

1. **Build**: Prototype your chain in a Jupyter Notebook.
2. **Trace**: Deploy it to staging with Tracing enabled. Ask it questions.
3. **Curate**: When the AI gives a bad answer, look at the Trace, fix the issue (e.g., prompt tweak), and save that user query to a LangSmith Dataset.
4. **Evaluate**: You now have a Dataset of 100 hard questions. Run your chain against it automatically on every pull request.
5. **Monitor**: Deploy to production. Monitor token costs and latency spikes.

---

# Part 12 — Preparing for LangGraph

## 12.1 The Limitations of LangChain

We have mastered LangChain. We can build complex, highly optimized Retrieval-Augmented Generation chains.

But look at the word **Chain**.

A Chain (`prompt | llm | output_parser`) is fundamentally a Directed Acyclic Graph (DAG). 
It goes strictly from Step A → Step B → Step C.

**What happens when the workflow requires a loop?**
- What if the LLM looks at the retrieved documents and says: *"These are not helpful. Let me rewrite the search query and try again."*
- A static Chain cannot do this. It cannot loop back.

**What happens when the workflow requires branching?**
- What if the LLM says: *"I need to search the web for this, but query a SQL database for that."*

**What happens when we need a Human in the Loop?**
- *"I am about to delete the user's database table. Stop and wait for the human to click 'Approve' before proceeding."*

## 12.2 Chains vs Agents

A **Chain** is a rigid, predetermined sequence of steps.
An **Agent** is an LLM that controls its own control flow. It has access to Tools (like our Retrievers, Web Search, Calculators). It decides *which* tool to use, *when* to use it, and *when* it has enough information to stop.

## 12.3 Enter LangGraph

To build resilient, autonomous Agents, LangChain created **LangGraph**.

LangGraph treats your AI application as a State Machine (a Graph).
- Nodes are Python functions or LLM calls.
- Edges contain the conditional logic (e.g., `if document_is_relevant == False: go_to_node("Search_Again")`).
- State is a shared memory object passed between all nodes.

LangGraph allows for cycles, long-running persistent agents, human-in-the-loop approvals, and multi-agent collaboration.

It is the future of AI Engineering.

**Next Notebook: LangGraph Fundamentals**
